# Mímir face à PyTorch, TensorFlow et JAX — benchmark reproductible

Ce notebook produit un **tableau de benchmark et une note mesurée pour Mímir**, sur CPU, CUDA ou **TPU-v5e1 / TPU-v6e1**.
Aucun classement n'est prérempli. Une installation indisponible, une opération incorrecte ou un repli CPU sur une cible accélérée reste une erreur visible, sans note.

Le premier tableau porte sur un ensemble déclaré d'opérations FP32 : Linear, Conv2d CHW, LayerNorm et un pas Linear + MSE + backward + SGD sans momentum.
Les entrées, poids, cibles et taux d'apprentissage sont identiques. Chaque pas SGD repart des mêmes poids : il mesure une étape, pas la convergence d'un modèle.
Ces microbenchmarks ne représentent ni toutes les architectures ni la qualité d'apprentissage d'un framework.
Deux tableaux indépendants : opérations élémentaires, puis entraînement et inférence MNIST sous contraintes effectives.


Détection automatique : laissez `BACKEND="auto"`. Le notebook sélectionne CUDA si un GPU NVIDIA est disponible, sinon TPU (v5e1/v6e1), sinon CPU. `"cpu"`, `"gpu"`/`"cuda"` ou `"tpu"` permettent de forcer le choix. Les dépendances, le build et les mesures utilisent ce même backend ; un accélérateur sélectionné mais non fonctionnel provoque un échec explicite.


Au début : `GPU_CUDA`, `GPU_VULKAN`, `GPU_OPENCL` activent indépendamment les runtimes **Mímir** (ignorés en CPU/TPU). Le benchmark réel laisse le planner choisir parmi ceux activés ; le microbenchmark mesure chaque runtime validé séparément : `mimir-cuda`, `mimir-vulkan`, `mimir-opencl`. Les concurrents ne sont mesurés qu’une fois par cas. Pour isoler Vulkan ou OpenCL, activer uniquement celui-ci. Les concurrents utilisent CUDA sur GPU. Les runtimes indisponibles (code 77) sont exclus et consignés dans le rapport ; `GPU_REQUIRE_ALL=True` rend leur absence bloquante. Au moins un runtime GPU doit être utilisable. Un échec numérique reste bloquant ; le pilote/ICD GPU doit être fourni par Colab, les outils installés ne le remplacent pas.


## Protocole et interprétation

- **Même instance et un seul accélérateur**, même budget CPU (affinité Linux et variables de threads), FP32, sans autocast/TF32 volontaire. Référence numérique indépendante NumPy en float64.
- **Frontière hôte identique** : les tableaux et poids commencent en RAM ; toutes les sorties sont ramenées en RAM avant d'arrêter le chronomètre. Les transferts, synchronisations et l'interface publique sont inclus. Le worker TPU de Mímir et son IPC sont donc mesurés. Ce tableau ne mesure pas des kernels résidents seuls.
- Modes affichés : Mímir dispatcher natif ; PyTorch eager (XLA sur TPU) ; TensorFlow `tf.function` ; JAX `jax.jit`. Il s'agit des chemins déclarés de chaque produit, pas d'une comparaison d'un même compilateur.
- Initialisation, premier appel/compilation et latences chaudes sont séparés. Warmup, essais indépendants et ordre aléatoire limitent les biais de démarrage et de température.
- Médiane, p95, IQR, débit, erreurs numériques et RSS hôte maximale sont exportés. La RSS est mesurée par `/usr/bin/time` dans chaque processus isolé, pour éviter un pic hérité du processus notebook. Elle inclut l'import des frameworks ; **elle ne mesure pas la VRAM/HBM** et ne participe pas à la note.
- Note par workload : `100 × meilleure_médiane / médiane_framework`. Note globale : moyenne géométrique de ces notes, à poids égaux. Seuls les frameworks corrects sur **tous les cas et essais** sont classés ; il faut au moins deux concurrents complets. Les erreurs ne valent jamais zéro milliseconde. Des écarts proches de l’IQR ne prouvent pas une différence significative.

Références de méthode : [JAX — benchmarking](https://docs.jax.dev/en/latest/benchmarking.html), [PyTorch — précision CUDA](https://docs.pytorch.org/docs/stable/notes/cuda), [TensorFlow — TF32](https://www.tensorflow.org/api_docs/python/tf/config/experimental/enable_tensor_float_32_execution).

Ce profil porte l’identifiant `mimir-colab-micro-v2` : comparer les notes uniquement entre exécutions du même profil et protocole.


## Configuration du matériel

In [ ]:
import os, sys, json, shutil, subprocess, time, platform, hashlib, signal
from pathlib import Path
from datetime import datetime, timezone
REPO_URL = "https://github.com/brit45/mimir-framework.git"
BRANCH = "develop"
REPO = Path("/content/mimir-framework")
BACKEND = "gpu"                    # auto | cpu | gpu/cuda | tpu
# Mímir GPU runtimes: independent switches (competitors retain CUDA).
GPU_CUDA = True
GPU_VULKAN = True
GPU_VULKAN_REPAIR = GPU_VULKAN
GPU_OPENCL = False
TPU_TARGET = "auto"                 # auto | TPU-v5e1 | TPU-v6e1
JOBS = min(2, os.cpu_count() or 1)
THREADS = min(2, os.cpu_count() or 1)
LOG_DIR = Path("/content/mimir-colab-logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def run(args, cwd=None, env=None, timeout=1800, capture=True, allowed_returncodes=()):
    command=list(map(str,args))
    log=LOG_DIR/(datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')+'.log')
    print('+', ' '.join(command), '\nJournal :',log,flush=True)
    # The child writes directly to a file: timeout remains effective even when
    # a compiler/driver stops producing stdout. Display the new log incrementally.
    with log.open('w') as output:
        p=subprocess.Popen(command,cwd=cwd,env=env,stdout=output,stderr=subprocess.STDOUT,text=True,start_new_session=True)
        started=time.monotonic();offset=0
        try:
            while True:
                with log.open(errors='replace') as reader:
                    reader.seek(offset);chunk=reader.read();offset=reader.tell()
                if chunk:print(chunk,end='',flush=True)
                if p.poll() is not None:break
                if time.monotonic()-started>timeout:
                    os.killpg(p.pid,signal.SIGKILL);p.wait();raise TimeoutError(f'Commande trop longue ; journal : {log}')
                time.sleep(.2)
        except BaseException:
            if p.poll() is None:os.killpg(p.pid,signal.SIGKILL);p.wait()
            raise
    text=log.read_text(errors='replace')
    if p.returncode and p.returncode not in allowed_returncodes:raise RuntimeError(f'Code {p.returncode} ; journal : {log}\n'+text[-12000:])
    return text

def choose_backend(request):
    # Probe hardware in children so the notebook never owns an accelerator.
    request=request.lower()
    if request=='gpu':request='cuda'
    if request not in {'auto','cpu','cuda','tpu'}:raise ValueError(request)
    if request!='auto':return request
    if shutil.which('nvidia-smi'):
        try:
            probe=subprocess.run(['nvidia-smi','--query-gpu=name,compute_cap','--format=csv,noheader'],
                                 capture_output=True,text=True,timeout=15)
            if probe.returncode==0 and probe.stdout.strip():return 'cuda'
            probe=subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True,timeout=15)
            if probe.returncode==0 and 'GPU' in probe.stdout:return 'cuda'
        except (OSError,subprocess.TimeoutExpired):pass
    if any(os.environ.get(k) for k in ('COLAB_TPU_ADDR','TPU_NAME','TPU_WORKER_ID','TPU_ACCELERATOR_TYPE')) or any(
            Path(p).exists() for p in ('/dev/accel0','/dev/accel/accel0')):return 'tpu'
    try:
        probe=subprocess.run([sys.executable,'-c',
            "import jax; print('MIMIR_TPU_COUNT='+str(len(jax.devices('tpu'))))"],
            capture_output=True,text=True,timeout=60)
        if probe.returncode==0 and any(line.startswith('MIMIR_TPU_COUNT=') and
                int(line.split('=',1)[1])>0 for line in probe.stdout.splitlines()):return 'tpu'
    except (OSError,ValueError,subprocess.TimeoutExpired):pass
    return 'cpu'
REQUESTED_BACKEND=BACKEND
BACKEND=choose_backend(BACKEND)
MIMIR_GPU_RUNTIMES=[name for name,enabled in [('cuda',GPU_CUDA),('vulkan',GPU_VULKAN),('opencl',GPU_OPENCL)] if enabled] if BACKEND=='cuda' else []
if BACKEND=='cuda' and not MIMIR_GPU_RUNTIMES:raise ValueError('Activer au moins un runtime GPU Mímir')
MIMIR_MICRO_RUNTIME=MIMIR_GPU_RUNTIMES[0] if MIMIR_GPU_RUNTIMES else BACKEND
BUILD=REPO/('build-colab-'+BACKEND+('-'+'-'.join(MIMIR_GPU_RUNTIMES) if MIMIR_GPU_RUNTIMES else ''))
RUNTIME_ENV=dict(os.environ, OMP_NUM_THREADS=str(THREADS), OPENBLAS_NUM_THREADS=str(THREADS),
    MKL_NUM_THREADS=str(THREADS), TF_NUM_INTRAOP_THREADS=str(THREADS), TF_NUM_INTEROP_THREADS='1',
    XLA_PYTHON_CLIENT_PREALLOCATE='false', TF_FORCE_GPU_ALLOW_GROWTH='true',
    XLA_USE_BF16='0', XLA_DOWNCAST_BF16='0',
    MIMIR_TPU='1' if BACKEND=='tpu' else '0', MIMIR_TPU_TARGET=TPU_TARGET,
    MIMIR_TPU_PYTHON=sys.executable, MIMIR_DISABLE_FPGA='1', MIMIR_DISABLE_ROCM='1',
    MIMIR_DISABLE_CUDA='0' if 'cuda' in MIMIR_GPU_RUNTIMES else '1',
    MIMIR_DISABLE_VULKAN='0' if 'vulkan' in MIMIR_GPU_RUNTIMES else '1', MIMIR_DISABLE_OPENCL='0' if 'opencl' in MIMIR_GPU_RUNTIMES else '1', MIMIR_BENCH_RUNTIME=MIMIR_MICRO_RUNTIME, MIMIR_ALLOCATOR_LOG='0')
# Make an explicit CPU selection effective even on an accelerator VM.
if BACKEND=='cpu':
    RUNTIME_ENV.update(CUDA_VISIBLE_DEVICES='',JAX_PLATFORMS='cpu',PJRT_DEVICE='CPU')
elif BACKEND=='cuda':
    RUNTIME_ENV.update(JAX_PLATFORMS='cuda',PJRT_DEVICE='CUDA')
else:
    RUNTIME_ENV.update(CUDA_VISIBLE_DEVICES='',JAX_PLATFORMS='tpu',PJRT_DEVICE='TPU')
ENVIRONMENT=dict(requested_backend=REQUESTED_BACKEND,
                 selected_backend=BACKEND,mimir_gpu_runtimes=MIMIR_GPU_RUNTIMES,mimir_micro_runtime=MIMIR_MICRO_RUNTIME,python=sys.version,platform=platform.platform())
print('Environnement detecte :',json.dumps(ENVIRONMENT,ensure_ascii=False))
print('Backend :',BACKEND,'| cible TPU :',TPU_TARGET,'| threads :',THREADS)

## Profil du benchmark
Le profil quick valide le système et donne une première mesure ; utiliser standard et davantage d’essais pour publier un classement.
Les profils incluent désormais BatchMatMul forward/backward et l’attention causale ; standard ajoute une attention de 2 048 tokens. Le backward matriciel mesure les deux produits de gradients, sans forward ni mise à jour des poids, dans chaque framework.


In [ ]:
# quick = 7 workloads; standard = 11 workloads. No dataset download required.
PROFILE='quick'
FRAMEWORKS=['mimir','pytorch','tensorflow','jax']
TRIALS=2 if PROFILE=='quick' else 3
ITERATIONS=12 if PROFILE=='quick' else 30
WARMUP=3 if PROFILE=='quick' else 5
CASE_TIMEOUT=300
SEED=20261017
ATOL,RTOL=3e-5,4e-4
assert PROFILE in {'quick','standard'} and TRIALS>=1 and ITERATIONS>=5 and WARMUP>=1
assert len(FRAMEWORKS)==len(set(FRAMEWORKS)) and 'mimir' in FRAMEWORKS
assert set(FRAMEWORKS)<={'mimir','pytorch','tensorflow','jax'}
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+os.urandom(3).hex()
RUN_DIR=Path('/content/mimir-bench-results')/RUN_ID
RUN_DIR.mkdir(parents=True,exist_ok=False)
LOG_DIR=RUN_DIR/'logs'
LOG_DIR.mkdir()
CPU_AFFINITY=sorted(os.sched_getaffinity(0))[:THREADS]
if not CPU_AFFINITY:raise RuntimeError('Affinité CPU vide')
print('Résultats :',RUN_DIR,'| cœurs CPU autorisés :',CPU_AFFINITY)

## Sources et dépendances Mímir
Les sources actuelles sont embarquées et vérifiées par SHA256, y compris les fichiers CUDA et TPU non encore publiés sur la branche distante. Extraction dans un dossier dédié, sans modification des sources CPU. `SOURCE_MODE="checkout"` active explicitement un checkout externe.

In [ ]:
# CPU setup must not inherit runtime requests from an earlier GPU session.
if BACKEND=='cpu':
    MIMIR_GPU_RUNTIMES=[]
    MIMIR_MICRO_RUNTIME='cpu'
    RUNTIME_ENV.update(MIMIR_BENCH_RUNTIME='cpu',MIMIR_DISABLE_CPU='0',MIMIR_TPU='0',
        MIMIR_DISABLE_CUDA='1',MIMIR_DISABLE_ROCM='1',MIMIR_DISABLE_VULKAN='1',
        MIMIR_DISABLE_OPENCL='1',CUDA_VISIBLE_DEVICES='',JAX_PLATFORMS='cpu',PJRT_DEVICE='CPU')
    if 'ENVIRONMENT' in globals():
        ENVIRONMENT.update(selected_backend='cpu',mimir_gpu_runtimes=[],mimir_micro_runtime='cpu')
apt=['apt-get'] if os.geteuid()==0 else ['sudo','apt-get']
run(apt+['update','-qq'])
run(apt+['install','-y','-qq','build-essential','g++-12','cmake','ninja-build','git',
         'pkg-config','util-linux','time','python3-venv','liblua5.3-dev','liblz4-dev','libssl-dev','nlohmann-json3-dev'])
# SDK/tools only: the accelerator driver/ICD must be supplied by the host.
if 'vulkan' in MIMIR_GPU_RUNTIMES:
    run(apt+['install','-y','-qq','libvulkan1','libvulkan-dev','vulkan-tools','glslang-tools','spirv-tools','binutils','libegl1','libglx0','libx11-6','libx11-xcb1','libxcb1','libxext6','libdrm2','libgbm1'])
if 'opencl' in MIMIR_GPU_RUNTIMES:
    run(apt+['install','-y','-qq','ocl-icd-opencl-dev','opencl-headers','clinfo'])
# BEGIN MIMIR EMBEDDED SOURCES
import base64, gzip, io, tarfile, tempfile
SOURCE_MODE = "embedded"  # embedded | checkout (explicit opt-in to REPO/BRANCH)
SOURCE_SHA256 = 'c1d35f1b04079187fa0d6fe8f364b98bbacd2344f81920bd9ea9232463b8abe6'
SOURCE_PAYLOAD = '...'
if SOURCE_MODE == 'embedded':
    packed = base64.b64decode(SOURCE_PAYLOAD, validate=True)
    if hashlib.sha256(packed).hexdigest() != SOURCE_SHA256:
        raise RuntimeError('Instantané des sources corrompu : recharger le notebook')
    # A dedicated immutable source directory preserves any existing checkout.
    snapshot_root = REPO.parent / ('mimir-source-' + SOURCE_SHA256[:16])
    def verify_snapshot(root, manifest):
        return all((root / name).is_file() and
                   hashlib.sha256((root / name).read_bytes()).hexdigest() == digest
                   for name, digest in manifest['files'].items())
    with tarfile.open(fileobj=io.BytesIO(gzip.decompress(packed)), mode='r:') as archive:
        members = archive.getmembers()
        for member in members:
            path = Path(member.name)
            if path.is_absolute() or '..' in path.parts or not member.isfile():
                raise RuntimeError('Chemin interdit dans les sources : ' + member.name)
        manifest = json.load(archive.extractfile('colab-source-manifest.json'))
        names = [m.name for m in members]
        if len(names) != len(set(names)) or set(names) != set(manifest['files']) | {'colab-source-manifest.json'}:
            raise RuntimeError('Inventaire des sources invalide')
        if snapshot_root.exists():
            if not verify_snapshot(snapshot_root, manifest):
                raise RuntimeError('Instantané local modifié : supprimer ce dossier puis relancer : ' + str(snapshot_root))
        else:
            snapshot_root.parent.mkdir(parents=True, exist_ok=True)
            with tempfile.TemporaryDirectory(prefix='mimir-extract-', dir=snapshot_root.parent) as temporary:
                temporary = Path(temporary)
                for member in members:
                    target = temporary / member.name
                    target.parent.mkdir(parents=True, exist_ok=True)
                    target.write_bytes(archive.extractfile(member).read())
                    target.chmod(member.mode)
                if not verify_snapshot(temporary, manifest):
                    raise RuntimeError('Empreintes des sources invalides')
                temporary.rename(snapshot_root)
    REPO = snapshot_root
    COMMIT = manifest['commit']
    DIRTY = manifest['provenance'] + '; snapshot_sha256=' + SOURCE_SHA256
elif SOURCE_MODE == 'checkout':
    if not REPO.exists():
        run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO])
    COMMIT = run(['git', 'rev-parse', 'HEAD'], cwd=REPO).strip()
    DIRTY = run(['git', 'status', '--short'], cwd=REPO)
else:
    raise ValueError('SOURCE_MODE doit être embedded ou checkout')
BUILD = REPO / f'build-colab-{BACKEND}'
print('Sources :', REPO, '| mode :', SOURCE_MODE)
# END MIMIR EMBEDDED SOURCES



















# Restore NVIDIA Vulkan userspace privately, matching the loaded host driver.
VULKAN_SETUP=dict(status='not_requested')
if 'vulkan' in MIMIR_GPU_RUNTIMES and GPU_VULKAN_REPAIR:
    # BEGIN MIMIR VULKAN HELPER
    import types
    vulkan_helper_source = '"""Private, exact-host-version NVIDIA Vulkan userspace for Colab.\n\nNever run nvidia-installer, install a kernel module, or replace system CUDA libs.\nOnly the NVIDIA official self-extractor is run with --extract-only.\n"""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport re\nimport shutil\nimport subprocess\nimport urllib.request\n\nGRAPHICS_PREFIXES=(\'libEGL_nvidia.so\',\'libGLX_nvidia.so\',\'libnvidia-eglcore.so\',\n                   \'libnvidia-glcore.so\',\'libnvidia-glsi.so\',\'libnvidia-gpucomp.so\',\n                   \'libnvidia-allocator.so\',\'libnvidia-tls.so\',\'libnvidia-glvkspirv.so\',\'libnvidia-vulkan-producer.so\')\n\n\ndef nvidia_device(output):\n    # Inspect physical-device blocks, rather than loader messages mentioning NVIDIA.\n    return any(re.search(r\'vendorID\\s*=\\s*(?:0x10de|4318)\\b\',block,re.I) and\n               re.search(r\'deviceType\\s*=\\s*PHYSICAL_DEVICE_TYPE_(?:DISCRETE|INTEGRATED)_GPU\',block)\n               for block in re.split(r\'GPU\\d+\\s*:\',output))\n\n\ndef probe(environment):\n    p=subprocess.run([\'vulkaninfo\',\'--summary\'],env=environment,capture_output=True,text=True,timeout=90)\n    return p.returncode==0 and nvidia_device(p.stdout),p.stdout+p.stderr\n\n\ndef stage_libraries(extracted,destination):\n    """Copy native x86_64 graphics ELF files and reconstruct their actual SONAMEs."""\n    destination.mkdir(parents=True,exist_ok=True)\n    copied=[]\n    for source in sorted(Path(extracted).rglob(\'*.so*\')):\n        if source.is_symlink() or not source.is_file() or not source.name.startswith(GRAPHICS_PREFIXES):continue\n        with source.open(\'rb\') as stream:header=stream.read(20)\n        if len(header)<20 or header[:5]!=b\'\\x7fELF\\x02\' or header[18:20]!=b\'\\x3e\\x00\':continue\n        target=destination/source.name\n        shutil.copy2(source,target)\n        dynamic=subprocess.run([\'readelf\',\'-d\',str(target)],capture_output=True,text=True,check=True)\n        soname=re.search(r\'\\(SONAME\\).*?\\[([^\\]]+)\\]\',dynamic.stdout)\n        if soname:\n            name=soname.group(1)\n            if Path(name).name!=name:raise RuntimeError(\'Invalid library SONAME\')\n            link=destination/name\n            if link!=target:\n                link.unlink(missing_ok=True);link.symlink_to(target.name)\n        copied.append(target.name)\n    if not any(name.startswith(\'libEGL_nvidia.so\') for name in copied):\n        raise RuntimeError(\'NVIDIA archive has no native libEGL_nvidia\')\n    return copied\n\n\ndef configure(environment,root,run):\n    """Return (child environment, audit report); callers keep availability policy."""\n    original=dict(environment)\n    report=dict(status=\'unavailable\',method=\'private NVIDIA graphics userspace\')\n    try:\n        ready,output=probe(original)\n        if ready:\n            report.update(status=\'host_ready\',probe=output)\n            return original,report\n        report[\'host_probe\']=output\n        if platform.machine()!=\'x86_64\':raise RuntimeError(\'Automatic NVIDIA extraction requires x86_64\')\n        version_output=run([\'nvidia-smi\',\'--query-gpu=driver_version\',\'--format=csv,noheader\'],timeout=30)\n        versions=set(version_output.strip().splitlines())\n        if len(versions)!=1:raise RuntimeError(\'Cannot determine a single NVIDIA host driver version\')\n        version=versions.pop().strip()\n        if not re.fullmatch(r\'\\d{3}\\.\\d{2}(?:\\.\\d{2})?\',version):raise RuntimeError(\'Invalid NVIDIA version: \'+version)\n        root=Path(root).resolve()/version;root.mkdir(parents=True,exist_ok=True)\n        archive=root/f\'NVIDIA-Linux-x86_64-{version}.run\'\n        url=f\'https://download.nvidia.com/XFree86/Linux-x86_64/{version}/{archive.name}\'\n        report.update(driver_version=version,url=url)\n        if not archive.is_file():\n            partial=archive.with_suffix(\'.partial\')\n            print(\'Téléchargement des composants NVIDIA officiels :\',url,flush=True)\n            try:\n                with urllib.request.urlopen(url,timeout=120) as response,partial.open(\'wb\') as target:\n                    shutil.copyfileobj(response,target)\n                partial.replace(archive)\n            finally:partial.unlink(missing_ok=True)\n        digest=hashlib.sha256()\n        with archive.open(\'rb\') as stream:\n            for chunk in iter(lambda:stream.read(1024*1024),b\'\'):digest.update(chunk)\n        report[\'archive_sha256\']=digest.hexdigest()\n        extracted=root/\'extracted\'\n        if not (extracted/\'.mimir-complete\').exists():\n            if extracted.exists():shutil.rmtree(extracted)\n            run([\'sh\',archive,\'--extract-only\',\'--target\',extracted],timeout=600)\n            (extracted/\'.mimir-complete\').touch()\n        library_dir=root/\'lib\'\n        report[\'libraries\']=stage_libraries(extracted,library_dir)\n        manifests=list(extracted.glob(\'nvidia_icd.json\'))\n        if not manifests:raise RuntimeError(\'Official NVIDIA ICD manifest absent from archive\')\n        icd=json.loads(manifests[0].read_text())\n        egl=next(library_dir/name for name in report[\'libraries\'] if name.startswith(\'libEGL_nvidia.so\'))\n        icd[\'ICD\'][\'library_path\']=str(egl)\n        manifest=root/\'nvidia_icd.json\';manifest.write_text(json.dumps(icd,indent=2))\n        candidate=dict(original,VK_DRIVER_FILES=str(manifest),VK_ICD_FILENAMES=str(manifest),\n                       LD_LIBRARY_PATH=str(library_dir)+(\':\'+original[\'LD_LIBRARY_PATH\'] if original.get(\'LD_LIBRARY_PATH\') else \'\'))\n        ready,output=probe(candidate)\n        report.update(probe=output,icd=str(manifest))\n        if not ready:raise RuntimeError(\'Private driver did not enumerate an NVIDIA Vulkan GPU; native tests remain authoritative\')\n        report[\'status\']=\'private_ready\'\n        return candidate,report\n    except Exception as exc:\n        report[\'error\']=str(exc)\n        print(\'Vulkan NVIDIA :\',str(exc),flush=True)\n        return original,report\n'
    vulkan_helper_sha256 = 'd3fe36f43d0a66be7ac6906f10feb9d423afc20378493658139658b5d10f5d23'
    if hashlib.sha256(vulkan_helper_source.encode()).hexdigest()!=vulkan_helper_sha256:raise RuntimeError('Module Vulkan embarque corrompu')
    vulkan_setup_module=types.ModuleType('mimir_nvidia_vulkan')
    exec(compile(vulkan_helper_source,'<mimir_nvidia_vulkan_embedded>','exec'),vulkan_setup_module.__dict__)
    # END MIMIR VULKAN HELPER
    RUNTIME_ENV,VULKAN_SETUP=vulkan_setup_module.configure(RUNTIME_ENV,Path('/content/mimir-nvidia-vulkan'),run)
ENVIRONMENT['vulkan_setup']=VULKAN_SETUP
(LOG_DIR/'vulkan-setup.json').write_text(json.dumps(VULKAN_SETUP,indent=2))
print('VULKAN_SETUP',VULKAN_SETUP.get('status'),VULKAN_SETUP.get('error',''))
# Vulkan loader/driver diagnostics: saved by run() even when no ICD is exposed.
if 'vulkan' in MIMIR_GPU_RUNTIMES:
    vulkan_probe = r"""
import glob,json,os,shutil,subprocess
paths=sorted(glob.glob('/usr/share/vulkan/icd.d/*.json')+glob.glob('/etc/vulkan/icd.d/*.json'))
print('VULKAN_ICD_FILES',json.dumps(paths),flush=True)
for path in paths:
    try:
        with open(path) as stream:print(path,stream.read(),flush=True)
    except OSError as exc:print(path,str(exc),flush=True)
commands=[['nvidia-smi'],['ldconfig','-p'],['vulkaninfo','--summary']]
for command in commands:
    print('DIAGNOSTIC',command,flush=True)
    if not shutil.which(command[0]):
        print('Commande absente',command[0],flush=True);continue
    env=dict(os.environ)
    if command[0]=='vulkaninfo':env['VK_LOADER_DEBUG']='all'
    try:
        p=subprocess.run(command,env=env,capture_output=True,text=True,timeout=60)
        output=p.stdout+p.stderr
        if command[0]=='ldconfig':output='\n'.join(line for line in output.splitlines() if any(name in line for name in ('libGLX_nvidia','libEGL_nvidia','libvulkan')))
        print(output,flush=True);print('DIAGNOSTIC_EXIT_CODE',p.returncode,flush=True)
    except (OSError,subprocess.TimeoutExpired) as exc:print(str(exc),flush=True)
"""
    run([sys.executable,'-c',vulkan_probe],env=RUNTIME_ENV,timeout=210)











required=['run_mimir.sh','src/runtimes/cpu/CpuRuntime.cpp','src/runtimes/RuntimeRouter.cpp',
          'src/runtimes/losses/LossDispatch.cpp','Tests/test_reconstruction_runtime.cpp']
if BACKEND=='tpu':required+=['src/runtimes/tpu/TpuRuntime.cpp','src/runtimes/tpu/worker.py',
                           'Tests/test_tpu_cpu_fixture.cpp','Tests/test_tpu_model_train.cpp']
if BACKEND=='cuda' and 'cuda' in MIMIR_GPU_RUNTIMES:required+=['src/runtimes/cuda/CudaNativeOps.cuh','Tests/test_cuda_native_ops.cpp']
missing=[p for p in required if not (REPO/p).is_file()]
if missing:raise RuntimeError('Sources incomplètes ; en mode checkout, utiliser embedded ou synchroniser la branche : '+str(missing))
CPU_HASHES={str(p.relative_to(REPO)):hashlib.sha256(p.read_bytes()).hexdigest()
            for p in (REPO/'src/runtimes/cpu').rglob('*') if p.is_file()}
if BACKEND=='tpu':run([sys.executable,'-m','pip','install','jax[tpu]'])
NVCC=shutil.which('nvcc') or ('/usr/local/cuda/bin/nvcc' if Path('/usr/local/cuda/bin/nvcc').is_file() else None)
if BACKEND=='cuda' and 'cuda' in MIMIR_GPU_RUNTIMES:
    if not NVCC:raise RuntimeError('Backend CUDA demandé, mais nvcc est absent')
    run(['nvidia-smi']);run([NVCC,'--version'])
BUILD.mkdir(parents=True,exist_ok=True)
CMAKE_ARGS=['cmake','-S',REPO,'-B',BUILD,'-G','Ninja','-DCMAKE_BUILD_TYPE=Release',
    '-DCMAKE_CXX_COMPILER=/usr/bin/g++-12','-DMIMIR_RUNTIME_OUTPUT_DIRECTORY='+str(BUILD/'bin'),
    '-DCMAKE_EXPORT_COMPILE_COMMANDS=ON','-DMIMIR_ENABLE_TESTS=ON','-DBUILD_TESTING=ON',
    '-DENABLE_CUDA='+('ON' if BACKEND=='cuda' and 'cuda' in MIMIR_GPU_RUNTIMES else 'OFF'),'-DENABLE_ROCM=OFF',
    '-DENABLE_VULKAN='+('ON' if BACKEND=='cuda' and 'vulkan' in MIMIR_GPU_RUNTIMES else 'OFF'),'-DENABLE_OPENCL='+('ON' if BACKEND=='cuda' and 'opencl' in MIMIR_GPU_RUNTIMES else 'OFF'),'-DMIMIR_AUTO_DETECT_GPU_BACKENDS=OFF',
    '-DMIMIR_VIZ_BACKEND=NONE','-DENABLE_SFML=OFF','-DENABLE_FFMPEG=OFF',
    '-DENABLE_SCRIPTING_RUST=OFF','-DENABLE_SCRIPTING_JS=OFF','-DENABLE_SCRIPTING_CSHARP=OFF',
    '-DMIMIR_PORTABLE_BUILD=OFF','-DENABLE_SIMD=ON','-DENABLE_OPENMP=ON']
if BACKEND=='cuda' and 'cuda' in MIMIR_GPU_RUNTIMES:CMAKE_ARGS+=['-DCMAKE_CUDA_HOST_COMPILER=/usr/bin/g++-12',f'-DCMAKE_CUDA_COMPILER={NVCC}']

## Environnements concurrents isolés
Les installations et imports ne sont pas chronométrés dans les latences chaudes. Un échec est enregistré ; les autres concurrents peuvent continuer.

PyTorch utilise une pile versionnée : `torch==2.9.0`, roues CPU ou CUDA 12.8 (compatible Blackwell) ; sur TPU, `torch_xla[tpu]==2.9.0` apporte le runtime libtpu correspondant. Python 3.10–3.13 est conservé ; sinon, un Python 3.12 dédié est installé via uv. Le noyau Colab reste intact. `pip check` et un calcul forward/backward sur le device demandé doivent passer avant les mesures.

Sources : [roues PyTorch](https://download.pytorch.org/whl/), [distribution PyTorch/XLA](https://pypi.org/project/torch-xla/2.9.0/).


In [ ]:
# Separate interpreters avoid package conflicts and accelerator ownership leaks.
VENV_ROOT=Path('/content/mimir-bench-envs')
PYTHONS={'mimir':sys.executable}
INSTALL_ERRORS={}
# Colab may run a Python distinct from the apt-installed python3-venv.
# virtualenv bundles a pip seeder for the actual kernel interpreter, and can
# repair existing environments left incomplete by a previous run.
run([sys.executable,'-m','pip','install','virtualenv'],timeout=600)
for framework in FRAMEWORKS:
    if framework=='mimir':continue
    try:
        bench_python=sys.executable
        if not ((3,11)<=sys.version_info[:2]<=(3,13)):
            # Use a common wheel ABI supported by the compared frameworks.
            run([sys.executable,'-m','pip','install','uv'],timeout=600)
            run([sys.executable,'-m','uv','python','install','3.12'],timeout=600)
            bench_python=run([sys.executable,'-m','uv','python','find','3.12'],timeout=60).strip().splitlines()[-1]
        suffix=('-torch2.9.0-cu128-v3' if BACKEND=='cuda' else '-torch2.9.0-v2') if framework=='pytorch' else ''
        directory=VENV_ROOT/(framework+'-'+BACKEND+suffix+'-py'+('kernel' if bench_python==sys.executable else '312'))
        interpreter=directory/'bin/python'
        pip_ready=False
        if interpreter.is_file():
            probe=subprocess.run([str(interpreter),'-m','pip','--version'],
                                 capture_output=True,text=True,timeout=30)
            pip_ready=probe.returncode==0
        if not pip_ready:
            run([sys.executable,'-m','virtualenv','--python',bench_python,directory],timeout=600)
        run([interpreter,'-m','pip','--version'],timeout=30)
        run([interpreter,'-m','pip','install','--upgrade','pip'],timeout=600)
        if framework=='pytorch':
            # torch_xla 2.9 is built against torch 2.9, not the latest torch.
            index='https://download.pytorch.org/whl/'+('cu128' if BACKEND=='cuda' else 'cpu')
            run([interpreter,'-m','pip','install','--upgrade','torch==2.9.0','--index-url',index],timeout=1800)
            run([interpreter,'-m','pip','install','numpy>=1.26,<3'],timeout=600)
            if BACKEND=='tpu':
                run([interpreter,'-m','pip','install','torch_xla[tpu]==2.9.0'],timeout=1800)
            run([interpreter,'-m','pip','check'],timeout=60)
            probe_code = """
import json,sys,torch,numpy as np
backend=sys.argv[1]
assert torch.__version__.split('+')[0]=='2.9.0',torch.__version__
if backend=='cuda':
    if not torch.cuda.is_available():raise RuntimeError('CUDA indisponible dans PyTorch; aucun repli CPU')
    if torch.version.cuda!='12.8':raise RuntimeError('Roues CUDA 12.8 attendues : '+str(torch.version.cuda))
    device=torch.device('cuda:0')
    print('PYTORCH_CUDA_DEVICE',torch.cuda.get_device_name(0),torch.cuda.get_device_capability(0),torch.cuda.get_arch_list())
elif backend=='tpu':
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.runtime as xr
    assert torch_xla.__version__.split('+')[0]=='2.9.0',torch_xla.__version__
    if xr.device_type()!='TPU':raise RuntimeError('PJRT ne fournit pas de TPU')
    device=xm.xla_device()
else:device=torch.device('cpu')
x=torch.from_numpy(np.ones((4,4),np.float32)).to(device).requires_grad_(True)
y=(x@x).sum();y.backward()
if backend=='tpu':xm.mark_step();xm.wait_device_ops()
if backend=='cuda':torch.cuda.synchronize()
assert float(y.detach().cpu())==64
assert np.allclose(x.grad.detach().cpu().numpy(),8)
print('PYTORCH_DEVICE_READY',json.dumps(dict(torch=torch.__version__,python=sys.version,device=str(device),cuda=torch.version.cuda)))
"""
            probe_env=dict(RUNTIME_ENV)
            if BACKEND=='tpu':probe_env['PJRT_DEVICE']='TPU'
            run([interpreter,'-I','-c',probe_code,BACKEND],env=probe_env,timeout=300)
        elif framework=='tensorflow':run([interpreter,'-m','pip','install',
                'tensorflow[and-cuda]' if BACKEND=='cuda' else 'tensorflow'],timeout=1800)
        elif framework=='jax':run([interpreter,'-m','pip','install',
                'jax[tpu]' if BACKEND=='tpu' else 'jax[cuda12]' if BACKEND=='cuda' else 'jax'],timeout=1800)
        PYTHONS[framework]=str(interpreter)
    except Exception as exc:
        INSTALL_ERRORS[framework]=str(exc)
        print(framework,'INSTALLATION_FAILED ; visible dans le tableau, pas de score')
run([sys.executable,'-m','pip','install','numpy','pandas','matplotlib'],timeout=600)

## Fixtures identiques et référence numérique

In [ ]:
import numpy as np
rng=np.random.default_rng(SEED)
def normal(shape,scale=.1):return (rng.standard_normal(shape)*scale).astype(np.float32)
def dense(case,n,i,o,train=False):
    return dict(case=case,op='linear_sgd' if train else 'linear',config=dict(n=n,i=i,o=o,lr=.01),
        x=normal((n,i)),weights=np.concatenate((normal((o,i),1/np.sqrt(i)).ravel(),normal((o,),.01))),
        target=normal((n,o)) if train else np.empty(0,np.float32),units=n,unit='vectors')
def conv(case,ci,co,h,w):
    k=3;return dict(case=case,op='conv2d',config=dict(ci=ci,co=co,h=h,w=w,k=k,pad=1,n=1,i=0,o=0),
        x=normal((ci,h,w)),weights=np.concatenate((normal((co,ci,k,k),.02).ravel(),normal((co,),.01))),
        target=np.empty(0,np.float32),units=1,unit='images')
def norm(case,n,i):
    return dict(case=case,op='layernorm',config=dict(n=n,i=i,o=i),x=normal((n,i)),
        weights=np.concatenate((1+normal((i,),.1),normal((i,),.01))),target=np.empty(0,np.float32),units=n,unit='tokens')
CASES=[dense('linear-small',32,128,64),conv('conv2d-chw',4,8,32,32),
       norm('layernorm',32,256),dense('linear-sgd',16,64,32,True)]
if PROFILE=='standard':CASES += [dense('linear-large',64,1024,1024),conv('conv2d-large',8,16,64,64)]

def batch_matmul(case,batch,m,k,n,backward=False):
    return dict(case=case,op='batch_matmul_backward' if backward else 'batch_matmul',
        config=dict(batch=batch,m=m,k=k,n=n),x=normal((batch,m,k)),
        weights=normal((batch,k,n)),target=normal((batch,m,n)) if backward else np.empty(0,np.float32),
        units=batch,unit='matrices')
def attention(case,seq,dim,heads,causal=False):
    return dict(case=case,op='self_attention',config=dict(seq=seq,dim=dim,heads=heads,causal=causal),
        x=normal((seq,dim)),weights=np.concatenate((normal((3*dim,dim),1/np.sqrt(dim)).ravel(),
        normal((dim,dim),1/np.sqrt(dim)).ravel())),target=np.empty(0,np.float32),units=seq,unit='tokens')
CASES += [batch_matmul('batch-matmul',2,128,64,128),
          batch_matmul('batch-matmul-backward',2,128,64,128,True),
          attention('attention-causal',129,32,4,True)]
if PROFILE=='standard':CASES += [batch_matmul('batch-matmul-large',8,128,256,192),
                               attention('attention-long',2048,128,4)]

def reference(f):
    c=f['config'];op=f['op'];x=np.asarray(f['x'],np.float64);p=np.asarray(f['weights'],np.float64)
    if op.startswith('batch_matmul'):
        a=x.reshape(c['batch'],c['m'],c['k']);b=p.reshape(c['batch'],c['k'],c['n'])
        if op=='batch_matmul':return [(a@b).ravel()]
        go=np.asarray(f['target'],np.float64).reshape(c['batch'],c['m'],c['n'])
        return [(go@b.swapaxes(-1,-2)).ravel(),(a.swapaxes(-1,-2)@go).ravel()]
    if op=='self_attention':
        seq,dim,heads=c['seq'],c['dim'],c['heads'];hd=dim//heads;qsize=3*dim*dim
        qkv=(x.reshape(seq,dim)@p[:qsize].reshape(3*dim,dim).T).reshape(seq,3,heads,hd)
        q,k,v=qkv.transpose(1,2,0,3);scores=(q@k.swapaxes(-1,-2))/np.sqrt(hd)
        if c['causal']:scores=np.where(np.arange(seq)[None,:]>np.arange(seq)[:,None],-1e9,scores)
        scores=np.exp(scores-scores.max(axis=-1,keepdims=True));scores/=scores.sum(axis=-1,keepdims=True)
        context=(scores@v).transpose(1,0,2).reshape(seq,dim)
        return [(context@p[qsize:].reshape(dim,dim)).ravel()]
    if op=='conv2d':
        ci,co,h,w,k=c['ci'],c['co'],c['h'],c['w'],c['k'];q=ci*co*k*k
        xx=np.pad(x.reshape(ci,h,w),((0,0),(c['pad'],)*2,(c['pad'],)*2))
        windows=np.lib.stride_tricks.sliding_window_view(xx,(k,k),axis=(1,2))
        y=np.einsum('chwkl,ockl->ohw',windows,p[:q].reshape(co,ci,k,k))+p[q:,None,None]
        return [y.ravel()]
    n,i,o=c['n'],c['i'],c['o'];xx=x.reshape(n,i)
    if op=='layernorm':
        z=xx-xx.mean(axis=-1,keepdims=True)
        return [(z/np.sqrt((z*z).mean(axis=-1,keepdims=True)+1e-5)*p[:i]+p[i:]).ravel()]
    q=i*o;w=p[:q].reshape(o,i);b=p[q:];y=xx@w.T+b
    if op=='linear':return [y.ravel()]
    d=y-np.asarray(f['target'],np.float64).reshape(n,o);loss=np.mean(d*d);go=2*d/d.size
    gw=np.concatenate(((go.T@xx).ravel(),go.sum(axis=0)))
    return [y.ravel(),np.array([loss]),p-c['lr']*gw,(go@w).ravel()]

GOLD={};FIXTURE_PATHS={}
for f in CASES:
    GOLD[f['case']]=reference(f)
    for key in ('x','weights','target'):f[key]=f[key].ravel().tolist()
    f.update(iterations=ITERATIONS,warmup=WARMUP)
    path=RUN_DIR/(f['case']+'.fixture.json');path.write_text(json.dumps(f,allow_nan=False))
    FIXTURE_PATHS[f['case']]=path
CASE_IDS=[f['case'] for f in CASES]
print('Charge totale par comparaison :',len(CASES),'cas ;',TRIALS,'processus par cas/framework ;',ITERATIONS,'mesures par processus')

## Pilote C++ Mímir et adaptateurs concurrents
Ces fichiers sont générés dans le build et le dossier de résultats. Aucun patch des runtimes n’est appliqué.

In [ ]:
CPP_SOURCE = '\n#include "Layers.hpp"\n#include "Model.hpp"\n#include "runtimes/optimizers/OptimizerDispatch.hpp"\n#include "runtimes/RuntimeRouter.hpp"\n#include "runtimes/cpu/CpuRuntime.hpp"\n#include "runtimes/tpu/TpuRuntime.hpp"\n#include <cstdlib>\n#ifdef ENABLE_VULKAN\n#include "runtimes/vulkan/VulkanRuntime.hpp"\n#endif\n#ifdef ENABLE_OPENCL\n#include "runtimes/opencl/OpenCLRuntime.hpp"\n#endif\n#ifdef ENABLE_CUDA\n#include "runtimes/cuda/CudaRuntime.hpp"\n#endif\n#include "include/json.hpp"\n#include <chrono>\n#include <fstream>\n#include <iostream>\n#include <memory>\n#include <stdexcept>\n#include <sys/resource.h>\nusing json=nlohmann::json;\nusing Clock=std::chrono::steady_clock;\nstatic double elapsed(Clock::time_point t){return std::chrono::duration<double,std::milli>(Clock::now()-t).count();}\nint main(int argc,char**argv){\n if(argc!=4)return 2;\n json report={{"framework","mimir"},{"status","error"}};\n try{\n  std::ifstream source(argv[2]);json f;source>>f;\n  const std::string backend=argv[1],op=f.at("op");const json c=f.at("config");\n  report["backend"]=backend;report["case"]=f.at("case");report["compiler"]=__VERSION__;\n  const std::string runtime_backend=std::getenv("MIMIR_BENCH_RUNTIME")?std::getenv("MIMIR_BENCH_RUNTIME"):backend;\n  RuntimeConfig cfg;cfg.backend=runtime_backend;cfg.linear_enabled=cfg.conv_enabled=cfg.norm_enabled=cfg.attention_enabled=true;\n  cfg.linear_min_ops=cfg.conv_min_ops=cfg.norm_min_elements=cfg.attention_min_ops=0;\n  std::unique_ptr<AbstractRuntime> rt;\n  auto startup=Clock::now();\n  if(runtime_backend=="cpu")rt=std::make_unique<CpuRuntime>();\n  else if(runtime_backend=="tpu")rt=std::make_unique<TpuRuntime>();\n#ifdef ENABLE_CUDA\n  else if(runtime_backend=="cuda")rt=std::make_unique<CudaRuntime>();\n#endif\n#ifdef ENABLE_VULKAN\n  else if(runtime_backend=="vulkan")rt=std::make_unique<VulkanRuntime>();\n#endif\n#ifdef ENABLE_OPENCL\n  else if(runtime_backend=="opencl")rt=std::make_unique<OpenCLRuntime>();\n#endif\n  else throw std::runtime_error("requested backend not compiled");\n  if(!rt->initialize(cfg))throw std::runtime_error("requested runtime initialization failed; no CPU fallback");\n  report["setup_ms"]=elapsed(startup);report["runtime"]=rt->name();\n  if(backend=="tpu")report["device"]=static_cast<TpuRuntime*>(rt.get())->deviceDescription();\n  const std::string expected=runtime_backend=="cpu"?"CPU":runtime_backend=="cuda"?"CUDA":runtime_backend=="vulkan"?"VULKAN":runtime_backend=="opencl"?"OPENCL":"TPU";\n  if(rt->name()!=expected)throw std::runtime_error("wrong runtime");\n  Layer layer("bench",op=="conv2d"?"Conv2d":op=="layernorm"?"LayerNorm":op=="self_attention"?"SelfAttention":op.find("batch_matmul")==0?"BatchMatMul":"Linear",0);\n  layer.use_bias=true;layer.affine=true;\n  if(op=="conv2d"){\n   layer.in_channels=c.at("ci");layer.out_channels=c.at("co");layer.input_height=c.at("h");layer.input_width=c.at("w");\n   layer.kernel_size=c.at("k");layer.stride=1;layer.padding=c.at("pad");\n  }else if(op.find("batch_matmul")==0){\n   layer.seq_len=c.at("batch");layer.in_features=c.at("m");layer.out_features=c.at("k");layer.embed_dim=c.at("n");\n  }else if(op=="self_attention"){\n   layer.seq_len=c.at("seq");layer.embed_dim=c.at("dim");layer.num_heads=c.at("heads");layer.causal=c.at("causal");\n  }else{layer.in_features=c.at("i");layer.out_features=c.at("o");layer.seq_len=c.at("n");layer.eps=1e-5f;}\n  layer.weights=f.at("weights").get<std::vector<float>>();\n  const auto x=f.at("x").get<std::vector<float>>();\n  const auto target=f.at("target").get<std::vector<float>>();\n  // A matrix operand is an input, not a parameter block. Keep it separate so\n  // portable runtime resource accounting does not count it twice.\n  const bool batched=op.find("batch_matmul")==0;\n  const auto rhs_values=batched?std::move(layer.weights):std::vector<float>{};\n  RuntimeRouter::instance().setRuntimes(nullptr,nullptr,nullptr,nullptr,rt.get());\n  std::vector<std::vector<float>> result;\n  auto step=[&]{\n   AbstractRuntime* selected=nullptr;std::vector<std::vector<float>> out;\n   const auto rhs=batched?&rhs_values:nullptr;\n   const std::vector<const std::vector<float>*> inputs=batched?\n       std::vector<const std::vector<float>*>{&x,rhs}:std::vector<const std::vector<float>*>{&x};\n   if(op=="batch_matmul_backward"){\n    if(!RuntimeRouter::instance().dispatchBackwardLayer(inputs,{&target},result,layer,true,&selected)||selected!=rt.get())\n     throw std::runtime_error("backward native route not selected");\n    return;\n   }\n   if(!RuntimeRouter::instance().dispatchForwardLayer(inputs,out,layer,op=="linear_sgd",&selected)||selected!=rt.get())\n    throw std::runtime_error("forward native route not selected");\n   if(op!="linear_sgd"){result=std::move(out);return;}\n   float loss=0;std::vector<float> go(out[0].size());\n   if(target.size()!=out[0].size())throw std::runtime_error("target shape");\n   for(size_t j=0;j<go.size();++j){const float d=out[0][j]-target[j];loss+=d*d;go[j]=2*d/go.size();}\n   loss/=go.size();layer.grad_weights.clear();layer.grad_bias.clear();std::vector<std::vector<float>> gi;\n   if(!RuntimeRouter::instance().dispatchBackwardLayer({&x},{&go},gi,layer,true,&selected)||selected!=rt.get())\n    throw std::runtime_error("backward native route not selected");\n   auto updated=layer.weights;const float lr=c.at("lr");\n   if(updated.size()!=layer.grad_weights.size())throw std::runtime_error("packed gradients missing");\n   Optimizer optimizer;optimizer.type=OptimizerType::SGD;optimizer.step=1;\n   RuntimeOptimizerRequest request{&optimizer,lr,updated.data(),layer.grad_weights.data(),nullptr,nullptr,updated.size()};\n   if(!RuntimeRouter::instance().dispatchOptimizer(request,&selected))throw std::runtime_error("optimizer runtime failed");\n   report["optimizer_runtime"]=selected->name();\n   result={std::move(out[0]),{loss},std::move(updated),std::move(gi[0])};\n  };\n  auto cold=Clock::now();step();report["cold_ms"]=elapsed(cold);\n  for(int j=0;j<f.at("warmup").get<int>();++j)step();\n  std::vector<double> timings;\n  for(int j=0;j<f.at("iterations").get<int>();++j){auto t=Clock::now();step();timings.push_back(elapsed(t));}\n  rusage usage{};getrusage(RUSAGE_SELF,&usage);\n  report["samples_ms"]=timings;report["outputs"]=result;report["peak_host_rss_mib"]=usage.ru_maxrss/1024.;\n  report["mode"]="native runtime dispatcher; host boundary";report["status"]="measured";\n  rt->shutdown();\n }catch(const std::exception&e){report["error"]=e.what();}\n std::ofstream out(argv[3]);out<<report.dump(2);return 0;\n}\n'

FRAMEWORK_WORKER = "\nimport os,sys,json,time,resource,traceback\nfrom pathlib import Path\nimport numpy as np\nframework,backend,fixture_file,result_file=sys.argv[1:]\nf=json.loads(Path(fixture_file).read_text());c=f['config'];op=f['op']\nx=np.asarray(f['x'],np.float32);p=np.asarray(f['weights'],np.float32);target=np.asarray(f['target'],np.float32)\nr={'framework':framework,'backend':backend,'case':f['case'],'status':'error'}\nstarted=time.perf_counter()\ndef unpack():\n    if op.startswith('batch_matmul'):\n        return x.reshape(c['batch'],c['m'],c['k']),p.reshape(c['batch'],c['k'],c['n']),np.empty(0,np.float32)\n    if op=='self_attention':\n        dim=c['dim'];q=3*dim*dim\n        return x.reshape(c['seq'],dim),p[:q].reshape(3*dim,dim),p[q:].reshape(dim,dim)\n    if op=='conv2d':\n        q=c['co']*c['ci']*c['k']**2\n        return x.reshape(1,c['ci'],c['h'],c['w']),p[:q].reshape(c['co'],c['ci'],c['k'],c['k']),p[q:]\n    if op=='layernorm':return x.reshape(c['n'],c['i']),p[:c['i']],p[c['i']:]\n    q=c['i']*c['o'];return x.reshape(c['n'],c['i']),p[:q].reshape(c['o'],c['i']),p[q:]\ntry:\n    a,w,b=unpack();truth=target.reshape(c['n'],c['o']) if op=='linear_sgd' else target\n    if op=='batch_matmul_backward':truth=target.reshape(c['batch'],c['m'],c['n'])\n    if framework=='pytorch':\n        if backend=='tpu':os.environ['PJRT_DEVICE']='TPU'\n        import torch\n        import torch.nn.functional as F\n        torch.set_num_threads(int(os.environ.get('OMP_NUM_THREADS','1')));torch.set_num_interop_threads(1)\n        torch.set_float32_matmul_precision('highest')\n        torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False;torch.backends.cudnn.benchmark=False\n        if backend=='cuda':\n            if not torch.cuda.is_available():raise RuntimeError('CUDA unavailable; no CPU fallback')\n            device=torch.device('cuda:0');device_name=torch.cuda.get_device_name(0)\n        elif backend=='tpu':\n            import torch_xla.core.xla_model as xm\n            import torch_xla.runtime as xr\n            device=xm.xla_device()\n            if xr.device_type()!='TPU':raise RuntimeError('XLA backend is not TPU')\n            device_name=str(xr.runtime_device_attributes());r['torch_xla_version']=__import__('torch_xla').__version__\n        else:device=torch.device('cpu');device_name='CPU'\n        r.update(version=torch.__version__,device=str(device),device_name=device_name,mode='eager' if backend!='tpu' else 'XLA lazy')\n        def step():\n            tensors=[torch.from_numpy(v.copy()).to(device) for v in (a,w,b)]\n            xx,ww,bb=tensors\n            if op=='linear_sgd':\n                for v in tensors:v.requires_grad_(True)\n            if any(v.device.type!=device.type for v in tensors):raise RuntimeError('wrong input device')\n            if op=='batch_matmul_backward':\n                go=torch.from_numpy(truth.copy()).to(device)\n                out=[go@ww.transpose(-1,-2),xx.transpose(-1,-2)@go]\n                return [v.detach().cpu().numpy().copy().reshape(-1) for v in out]\n            if op=='batch_matmul':y=xx@ww\n            elif op=='self_attention':\n                seq,dim,heads=c['seq'],c['dim'],c['heads'];hd=dim//heads\n                q,k,v=(xx@ww.T).reshape(seq,3,heads,hd).permute(1,2,0,3).unbind(0)\n                scores=(q@k.transpose(-1,-2))/(hd**.5)\n                if c['causal']:\n                    indices=torch.arange(seq,device=device);scores=scores.masked_fill(indices[None,:]>indices[:,None],-1e9)\n                y=(torch.softmax(scores,dim=-1)@v).transpose(0,1).reshape(seq,dim)@bb\n            elif op=='conv2d':y=F.conv2d(xx,ww,bb,padding=c['pad'])\n            elif op=='layernorm':y=F.layer_norm(xx,(c['i'],),ww,bb,1e-5)\n            else:y=F.linear(xx,ww,bb)\n            if y.device.type!=device.type:raise RuntimeError('wrong output device')\n            out=[y]\n            if op=='linear_sgd':\n                yy=torch.from_numpy(truth.copy()).to(device);loss=((y-yy)**2).mean()\n                dx,dw,db=torch.autograd.grad(loss,(xx,ww,bb))\n                with torch.no_grad():new=torch.cat(((ww-c['lr']*dw).reshape(-1),(bb-c['lr']*db).reshape(-1)))\n                out=[y,loss.reshape(1),new,dx]\n            # Read every output on the host: synchronizes CUDA and XLA.\n            return [v.detach().cpu().numpy().copy().reshape(-1) for v in out]\n    elif framework=='jax':\n        import jax\n        import jax.numpy as jnp\n        from jax import lax\n        jax.config.update('jax_default_matmul_precision','highest')\n        device=jax.devices({'cpu':'cpu','cuda':'gpu','tpu':'tpu'}[backend])[0]\n        wanted={'cpu':'cpu','cuda':'gpu','tpu':'tpu'}[backend]\n        if device.platform!=wanted:raise RuntimeError('wrong JAX platform')\n        r.update(version=jax.__version__,device=str(device),device_name=device.device_kind,mode='jax.jit')\n        def forward(xx,ww,bb):\n            if op=='batch_matmul':return xx@ww\n            if op=='self_attention':\n                seq,dim,heads=c['seq'],c['dim'],c['heads'];hd=dim//heads\n                q,k,v=(xx@ww.T).reshape(seq,3,heads,hd).transpose(1,2,0,3)\n                scores=(q@jnp.swapaxes(k,-1,-2))/(hd**.5)\n                if c['causal']:scores=jnp.where(jnp.arange(seq)[None,:]>jnp.arange(seq)[:,None],-1e9,scores)\n                return (jax.nn.softmax(scores,axis=-1)@v).transpose(1,0,2).reshape(seq,dim)@bb\n            if op=='conv2d':return lax.conv_general_dilated(xx,ww,(1,1),((c['pad'],c['pad']),)*2,\n                dimension_numbers=('NCHW','OIHW','NCHW'),precision=lax.Precision.HIGHEST)+bb[None,:,None,None]\n            if op=='layernorm':\n                z=xx-jnp.mean(xx,axis=-1,keepdims=True)\n                return z*lax.rsqrt(jnp.mean(z*z,axis=-1,keepdims=True)+1e-5)*ww+bb\n            return xx@ww.T+bb\n        def kernel(xx,ww,bb,tt):\n            if op=='batch_matmul_backward':return tt@jnp.swapaxes(ww,-1,-2),jnp.swapaxes(xx,-1,-2)@tt\n            if op!='linear_sgd':return (forward(xx,ww,bb),)\n            def objective(v,u,z):\n                y=forward(v,u,z);return jnp.mean((y-tt)**2),y\n            (loss,y),(dx,dw,db)=jax.value_and_grad(objective,argnums=(0,1,2),has_aux=True)(xx,ww,bb)\n            return y,loss.reshape(1),jnp.concatenate(((ww-c['lr']*dw).ravel(),(bb-c['lr']*db).ravel())),dx\n        compiled=jax.jit(kernel)\n        def step():\n            args=[jax.device_put(v.copy(),device) for v in (a,w,b,truth)]\n            with jax.default_device(device):out=compiled(*args)\n            if any(next(iter(v.devices())).platform!=wanted for v in out):raise RuntimeError('wrong JAX output device')\n            return [np.asarray(v).copy().reshape(-1) for v in out]\n    elif framework=='tensorflow':\n        import tensorflow as tf\n        tf.config.set_soft_device_placement(False)\n        tf.config.threading.set_intra_op_parallelism_threads(int(os.environ.get('OMP_NUM_THREADS','1')))\n        tf.config.threading.set_inter_op_parallelism_threads(1)\n        tf.config.experimental.enable_tensor_float_32_execution(False)\n        if backend=='cuda':\n            gpus=tf.config.list_physical_devices('GPU')\n            if not gpus:raise RuntimeError('TensorFlow GPU unavailable; no CPU fallback')\n            for gpu in gpus:tf.config.experimental.set_memory_growth(gpu,True)\n            device='/GPU:0';device_name=str(tf.config.experimental.get_device_details(gpus[0]))\n        elif backend=='tpu':\n            resolver=tf.distribute.cluster_resolver.TPUClusterResolver(tpu='local')\n            tf.tpu.experimental.initialize_tpu_system(resolver)\n            devices=tf.config.list_logical_devices('TPU')\n            if len(devices)!=1:raise RuntimeError('one TPU device required')\n            device=devices[0].name;device_name=str(devices)\n            strategy=tf.distribute.TPUStrategy(resolver)\n            if any('TPU' not in d.upper() for d in strategy.extended.worker_devices):raise RuntimeError('TPUStrategy not placed on TPU')\n        else:\n            tf.config.set_visible_devices([],'GPU');device='/CPU:0';device_name='CPU'\n        r.update(version=tf.__version__,device=device,device_name=device_name,mode='tf.function + TPUStrategy' if backend=='tpu' else 'tf.function')\n        def forward(xx,ww,bb):\n            if op=='batch_matmul':return tf.linalg.matmul(xx,ww)\n            if op=='self_attention':\n                seq,dim,heads=c['seq'],c['dim'],c['heads'];hd=dim//heads\n                qkv=tf.reshape(tf.linalg.matmul(xx,ww,transpose_b=True),(seq,3,heads,hd))\n                q,k,v=tf.unstack(tf.transpose(qkv,(1,2,0,3)),axis=0)\n                scores=tf.linalg.matmul(q,k,transpose_b=True)/(hd**.5)\n                if c['causal']:scores=tf.where(tf.range(seq)[None,:]>tf.range(seq)[:,None],-1e9,scores)\n                context=tf.reshape(tf.transpose(tf.linalg.matmul(tf.nn.softmax(scores,axis=-1),v),(1,0,2)),(seq,dim))\n                return tf.linalg.matmul(context,bb)\n            if op=='conv2d':\n                xx=tf.transpose(xx,(0,2,3,1));ww=tf.transpose(ww,(2,3,1,0))\n                return tf.transpose(tf.nn.bias_add(tf.nn.conv2d(xx,ww,strides=1,padding='SAME' if c['pad'] else 'VALID'),bb),(0,3,1,2))\n            if op=='layernorm':\n                mean,var=tf.nn.moments(xx,axes=[-1],keepdims=True)\n                return (xx-mean)*tf.math.rsqrt(var+1e-5)*ww+bb\n            return tf.linalg.matmul(xx,ww,transpose_b=True)+bb\n        def kernel(xx,ww,bb,tt):\n            if op=='batch_matmul_backward':return tf.linalg.matmul(tt,ww,transpose_b=True),tf.linalg.matmul(xx,tt,transpose_a=True)\n            if op!='linear_sgd':return (forward(xx,ww,bb),)\n            with tf.GradientTape() as tape:\n                tape.watch((xx,ww,bb));y=forward(xx,ww,bb);loss=tf.reduce_mean((y-tt)**2)\n            dx,dw,db=tape.gradient(loss,(xx,ww,bb))\n            new=tf.concat((tf.reshape(ww-c['lr']*dw,[-1]),tf.reshape(bb-c['lr']*db,[-1])),0)\n            return y,tf.reshape(loss,[1]),new,dx\n        if backend=='tpu':\n            compiled=tf.function(lambda xx,ww,bb,tt:strategy.run(kernel,args=(xx,ww,bb,tt)),autograph=False)\n        else:compiled=tf.function(kernel,autograph=False)\n        placement_checked=False\n        def step():\n            global placement_checked\n            with tf.device('/CPU:0' if backend=='tpu' else device):\n                args=[tf.convert_to_tensor(v.copy(),dtype=tf.float32) for v in (a,w,b,truth)]\n                out=compiled(*args)\n            if backend=='tpu' and not placement_checked:\n                graph=compiled.get_concrete_function(*args).graph.as_graph_def()\n                nodes=list(graph.node)+[n for f in graph.library.function for n in f.node_def]\n                if not any('TPU' in n.op or '_tpu_replicate' in n.attr for n in nodes):raise RuntimeError('TPU execution graph proof missing')\n                placement_checked=True;r['placement_proof']='TPUStrategy + TPU graph nodes'\n            if backend=='tpu':out=tuple(strategy.experimental_local_results(v)[0] for v in out)\n            expected='GPU' if backend=='cuda' else 'TPU' if backend=='tpu' else 'CPU'\n            if backend!='tpu' and any(expected not in v.device.upper() for v in out):raise RuntimeError('TensorFlow output placed on wrong backend')\n            return [v.numpy().copy().reshape(-1) for v in out]\n    else:raise ValueError(framework)\n    r['setup_ms']=(time.perf_counter()-started)*1000\n    start=time.perf_counter_ns();out=step();r['cold_ms']=(time.perf_counter_ns()-start)/1e6\n    for _ in range(f['warmup']):step()\n    samples=[]\n    for _ in range(f['iterations']):\n        start=time.perf_counter_ns();out=step();samples.append((time.perf_counter_ns()-start)/1e6)\n    r.update(status='measured',samples_ms=samples,outputs=[v.tolist() for v in out],\n             peak_host_rss_mib=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024)\nexcept Exception as exc:r.update(error=str(exc),traceback=traceback.format_exc())\nPath(result_file).write_text(json.dumps(r,allow_nan=False))\n"


In [ ]:
TEST_TARGETS=['RuntimeTest.MathLinear','RuntimeTest.MathConv2d','RuntimeTest.MathNorms',
              'RuntimeTest.MathAttention','RuntimeTest.NMS','AutogradTest.RuntimeDispatch','RuntimeTest.OptimizerDispatch','RuntimeTest.ReconstructionLoss','ModelTest.ReconLosses','ModelTest.ExtendedOptimizers']
# Core allocation, tiled matrix and attention regressions (CPU reference).
TEST_TARGETS += ['RuntimeTest.LinearBackwardTiled', 'RuntimeTest.TypedConversions', 'RuntimeTest.ConvolutionScratch', 'RuntimeTest.AllocatorScratch', 'RuntimeTest.MatMulBackward', 'RuntimeTest.BatchMatMulForward', 'RuntimeTest.FlatJoin', 'RuntimeTest.FiniteCheck', 'RuntimeTest.RouteRefresh', 'RuntimeTest.AttentionTiled']
TEST_TARGETS += ['RuntimeTest.OptimizerSelection']
TEST_TARGETS += ['FrameworkTest.ExecutionCalibration', 'FrameworkTest.ExecutionPlan', 'FrameworkTest.PlannerEquivalence', 'FrameworkTest.PlannerLifetimes', 'FrameworkTest.PlannerFusionAndScratch', 'ModelTest.VGG16FeatEncNorm', 'VizTest.AnnotationFeatureBounds', 'RuntimeTest.OptimizerCompact']
if 'cuda' in MIMIR_GPU_RUNTIMES:TEST_TARGETS+=['RuntimeTest.CudaNativeOps']
if any(r in MIMIR_GPU_RUNTIMES for r in ('vulkan','opencl')):TEST_TARGETS+=['PortableNativeOps','RuntimeTest.PortableWorkBudget']
if BACKEND=='tpu':TEST_TARGETS+=['RuntimeTest.TpuCpuFixture','RuntimeTest.TpuModelTrain']


# Generate only the driver in the build directory; Mímir sources are read-only.
DRIVER=BUILD/'colab-bench-driver.cpp';DRIVER.write_text(CPP_SOURCE)
WORKER_PATH=RUN_DIR/'framework_worker.py';WORKER_PATH.write_text(FRAMEWORK_WORKER)
INJECT=BUILD/'colab-bench.cmake'
INJECT.write_text(r"""
function(mimir_colab_add_benchmark)
  if(TARGET mimir_core AND NOT TARGET mimir_colab_bench)
    add_executable(mimir_colab_bench "${CMAKE_BINARY_DIR}/colab-bench-driver.cpp")
    target_compile_features(mimir_colab_bench PRIVATE cxx_std_17)
    target_link_libraries(mimir_colab_bench PRIVATE mimir_core)
    set_target_properties(mimir_colab_bench PROPERTIES RUNTIME_OUTPUT_DIRECTORY "${CMAKE_BINARY_DIR}/bin")
  endif()
endfunction()
cmake_language(DEFER CALL mimir_colab_add_benchmark)
""")
run(CMAKE_ARGS+['-DCMAKE_PROJECT_INCLUDE='+str(INJECT)])
run(['cmake','--build',BUILD,'-j',JOBS,'--target','mimir_colab_bench',*TEST_TARGETS],timeout=2400)
BENCH_BINARY=BUILD/'bin/mimir_colab_bench'
if not BENCH_BINARY.is_file():raise RuntimeError(f'Binaire absent : {BENCH_BINARY}')

## Validation préalable du runtime Mímir

In [ ]:
GPU_REQUIRE_ALL = globals().get("GPU_REQUIRE_ALL", False)
VALIDATED=False
# Clear GPU state when this cell is resumed after switching to CPU.
if BACKEND=='auto':
    MIMIR_GPU_RUNTIMES=[]
    MIMIR_MICRO_RUNTIME='auto'
    RUNTIME_ENV.update(MIMIR_BENCH_RUNTIME='cpu',MIMIR_DISABLE_CPU='0',MIMIR_TPU='0',
        MIMIR_DISABLE_CUDA='0',MIMIR_DISABLE_ROCM='1',MIMIR_DISABLE_VULKAN='1',
        MIMIR_DISABLE_OPENCL='1',CUDA_VISIBLE_DEVICES='',JAX_PLATFORMS='cpu',PJRT_DEVICE='CPU')
# These analytical tests also contain compiled-in GPU branches. Build the
# reference without GPU backends so it cannot activate them implicitly.
REFERENCE_TARGETS=['RuntimeTest.MathLinear','RuntimeTest.MathConv2d','RuntimeTest.MathNorms',
                   'RuntimeTest.MathAttention','RuntimeTest.NMS','AutogradTest.RuntimeDispatch','RuntimeTest.OptimizerDispatch','RuntimeTest.ReconstructionLoss','ModelTest.ReconLosses','ModelTest.ExtendedOptimizers']
REFERENCE_TARGETS += ['RuntimeTest.LinearBackwardTiled', 'RuntimeTest.TypedConversions', 'RuntimeTest.ConvolutionScratch', 'RuntimeTest.AllocatorScratch', 'RuntimeTest.MatMulBackward', 'RuntimeTest.BatchMatMulForward', 'RuntimeTest.FlatJoin', 'RuntimeTest.FiniteCheck', 'RuntimeTest.RouteRefresh', 'RuntimeTest.AttentionTiled']
REFERENCE_TARGETS += ['RuntimeTest.OptimizerSelection']
REFERENCE_TARGETS += ['FrameworkTest.ExecutionCalibration', 'FrameworkTest.ExecutionPlan', 'FrameworkTest.PlannerEquivalence', 'FrameworkTest.PlannerLifetimes', 'FrameworkTest.PlannerFusionAndScratch', 'ModelTest.VGG16FeatEncNorm', 'VizTest.AnnotationFeatureBounds', 'RuntimeTest.OptimizerCompact']
REFERENCE_BUILD=BUILD
REFERENCE_ENV=dict(RUNTIME_ENV,MIMIR_TPU='0',MIMIR_DISABLE_CPU='0',
    MIMIR_DISABLE_CUDA='1',MIMIR_DISABLE_ROCM='1',MIMIR_DISABLE_VULKAN='1',MIMIR_DISABLE_OPENCL='1')
# An existing GPU build must not become the CPU reference after a mode switch.
GPU_BUILD_OPTIONS=('-DENABLE_CUDA=','-DENABLE_ROCM=','-DENABLE_VULKAN=',
                   '-DENABLE_OPENCL=','-DMIMIR_AUTO_DETECT_GPU_BACKENDS=')
REFERENCE_HAS_GPU=any(str(arg).startswith(GPU_BUILD_OPTIONS) and
                      str(arg).split('=',1)[1].upper() in {'ON','TRUE','YES','1'} for arg in CMAKE_ARGS)
if BACKEND!='cpu' or REFERENCE_HAS_GPU:
    REFERENCE_BUILD=REPO/'build-colab-reference-cpu'
    reference_args=[]
    for arg in CMAKE_ARGS:
        value=str(arg)
        if value.startswith(('-DCMAKE_CUDA_','-DCMAKE_PROJECT_INCLUDE=')):continue
        if value.startswith(GPU_BUILD_OPTIONS):value=value.split('=',1)[0]+'=OFF'
        if value.startswith('-DMIMIR_RUNTIME_OUTPUT_DIRECTORY='):value='-DMIMIR_RUNTIME_OUTPUT_DIRECTORY='+str(REFERENCE_BUILD/'bin')
        reference_args.append(value)
    reference_args[reference_args.index('-B')+1]=str(REFERENCE_BUILD)
    run(reference_args+['-DCMAKE_PROJECT_INCLUDE='],timeout=600)
    run(['cmake','--build',REFERENCE_BUILD,'-j',JOBS,'--target',*REFERENCE_TARGETS],timeout=2400)
run(['ctest','--test-dir',REFERENCE_BUILD,'--output-on-failure','--no-tests=error','-R',
     r'^(FrameworkTest.(ExecutionCalibration|ExecutionPlan|PlannerEquivalence|PlannerLifetimes|PlannerFusionAndScratch)|VizTest.AnnotationFeatureBounds|ModelTest.VGG16FeatEncNorm|RuntimeTest\.(MathLinear|MathConv2d|MathNorms|MathAttention|NMS|OptimizerDispatch|OptimizerSelection|OptimizerCompact|ReconstructionLoss|LinearBackwardTiled|TypedConversions|ConvolutionScratch|AllocatorScratch|MatMulBackward|BatchMatMulForward|FlatJoin|FiniteCheck|RouteRefresh|AttentionTiled)|AutogradTest.RuntimeDispatch|ModelTest.ReconLosses|ModelTest.ExtendedOptimizers)$'],
    cwd=REPO,env=REFERENCE_ENV,timeout=600)
if 'cuda' in MIMIR_GPU_RUNTIMES:
    cuda_validation_env=dict(REFERENCE_ENV,MIMIR_DISABLE_CUDA='0')
    output=run([BUILD/'Tests/RuntimeTest.CudaNativeOps'],cwd=REPO,env=cuda_validation_env,timeout=600)
    if 'CUDA_NATIVE_PARITY_OK' not in output or 'SKIP:' in output:raise RuntimeError('CUDA natif non validé')
    output=run([BUILD/'Tests/RuntimeTest.OptimizerDispatch','CUDA'],cwd=REPO,env=cuda_validation_env,timeout=600)
    if 'OPTIMIZER_RUNTIME_OK CUDA' not in output:raise RuntimeError('Optimizers CUDA non valides')
    output=run([BUILD/'Tests/RuntimeTest.ReconstructionLoss','CUDA'],cwd=REPO,env=cuda_validation_env,timeout=600)
    if 'RECONSTRUCTION_RUNTIME_OK CUDA' not in output:raise RuntimeError('Loss de reconstruction CUDA non valides')
if any(r in MIMIR_GPU_RUNTIMES for r in ('vulkan','opencl')):
    run([BUILD/'Tests/RuntimeTest.PortableWorkBudget'],cwd=REPO,env=REFERENCE_ENV,timeout=60)
# Compiled support and usable host drivers are separate capabilities.
REQUESTED_GPU_RUNTIMES=list(MIMIR_GPU_RUNTIMES)
UNAVAILABLE_GPU_RUNTIMES={}
ACTIVE_GPU_RUNTIMES=[]
for runtime in REQUESTED_GPU_RUNTIMES:
    if runtime=='cuda':
        ACTIVE_GPU_RUNTIMES.append(runtime)  # CUDA parity passed above.
        continue
    runtime_validation_env=dict(REFERENCE_ENV)
    runtime_validation_env['MIMIR_DISABLE_'+runtime.upper()]='0'
    output=run([BUILD/'Tests/PortableNativeOps',runtime.upper()],cwd=REPO,env=runtime_validation_env,
               timeout=600,allowed_returncodes=(77,))
    if 'SKIP:' in output:
        UNAVAILABLE_GPU_RUNTIMES[runtime]=output[-12000:]
        RUNTIME_ENV['MIMIR_DISABLE_'+runtime.upper()]='1'
        print(runtime.upper(),'indisponible sur cet hôte ; exclu des mesures GPU')
        if globals().get('GPU_REQUIRE_ALL', False):raise RuntimeError(runtime+' demandé mais indisponible ; GPU_REQUIRE_ALL=True')
        continue
    if 'GPU_NATIVE_PARITY_OK' not in output:raise RuntimeError(runtime+' natif non validé')
    output=run([BUILD/'Tests/RuntimeTest.OptimizerDispatch',runtime.upper()],cwd=REPO,env=runtime_validation_env,timeout=600,allowed_returncodes=(77,))
    if 'SKIP:' in output:
        UNAVAILABLE_GPU_RUNTIMES[runtime]='optimizer: '+output[-12000:]
        RUNTIME_ENV['MIMIR_DISABLE_'+runtime.upper()]='1'
        print(runtime.upper(),'optimizer GPU indisponible ; runtime exclu des mesures')
        if globals().get('GPU_REQUIRE_ALL', False):raise RuntimeError(runtime+' optimizer GPU indisponible ; GPU_REQUIRE_ALL=True')
        continue
    if 'OPTIMIZER_RUNTIME_OK '+runtime.upper() not in output:raise RuntimeError('Optimizers '+runtime+' non valides')
    output=run([BUILD/'Tests/RuntimeTest.ReconstructionLoss',runtime.upper()],cwd=REPO,env=runtime_validation_env,timeout=600,allowed_returncodes=(77,))
    if 'SKIP:' in output:
        UNAVAILABLE_GPU_RUNTIMES[runtime]='reconstruction_loss: '+output[-12000:]
        RUNTIME_ENV['MIMIR_DISABLE_'+runtime.upper()]='1'
        if globals().get('GPU_REQUIRE_ALL', False):raise RuntimeError(runtime+' loss GPU indisponible ; GPU_REQUIRE_ALL=True')
        continue
    if 'RECONSTRUCTION_RUNTIME_OK '+runtime.upper() not in output:raise RuntimeError('Loss de reconstruction '+runtime+' non valides')
    ACTIVE_GPU_RUNTIMES.append(runtime)
if BACKEND=='cuda':
    if not ACTIVE_GPU_RUNTIMES:raise RuntimeError('Aucun runtime GPU Mímir utilisable ; activer CUDA ou fournir un pilote/ICD Vulkan/OpenCL. Aucun repli CPU implicite.')
    MIMIR_GPU_RUNTIMES=ACTIVE_GPU_RUNTIMES
    MIMIR_MICRO_RUNTIME=ACTIVE_GPU_RUNTIMES[0]
    RUNTIME_ENV['MIMIR_BENCH_RUNTIME']=MIMIR_MICRO_RUNTIME
ENVIRONMENT.update(mimir_gpu_runtimes_requested=REQUESTED_GPU_RUNTIMES,
                   mimir_gpu_runtimes=list(MIMIR_GPU_RUNTIMES),mimir_micro_runtime=MIMIR_MICRO_RUNTIME,
                   unavailable_gpu_runtimes=UNAVAILABLE_GPU_RUNTIMES)
if BACKEND=='tpu':
    output=run([BUILD/'Tests/RuntimeTest.TpuCpuFixture','--tpu'],cwd=REPO,env=RUNTIME_ENV,timeout=1800)
    if 'TPU_HARDWARE_PARITY_OK' not in output:raise RuntimeError('Parité matérielle TPU non validée')
    output=run([BUILD/'Tests/RuntimeTest.TpuModelTrain'],cwd=REPO,env=RUNTIME_ENV,timeout=600)
    if 'TPU_MODEL_TRAIN_OK' not in output:raise RuntimeError('Entraînement natif Mímir/TPU non validé')
    output=run([BUILD/'Tests/RuntimeTest.OptimizerDispatch','TPU'],cwd=REPO,env=RUNTIME_ENV,timeout=1800)
    if 'OPTIMIZER_RUNTIME_OK TPU' not in output:raise RuntimeError('Optimizers TPU non valides')
    output=run([BUILD/'Tests/RuntimeTest.ReconstructionLoss','TPU'],cwd=REPO,env=RUNTIME_ENV,timeout=1800)
    if 'RECONSTRUCTION_RUNTIME_OK TPU' not in output:raise RuntimeError('Loss de reconstruction TPU non valides')
assert CPU_HASHES=={str(p.relative_to(REPO)):hashlib.sha256(p.read_bytes()).hexdigest()
                   for p in (REPO/'src/runtimes/cpu').rglob('*') if p.is_file()}, 'Sources CPU modifiées'
VALIDATED=True
print('VALIDATION_OK',BACKEND,COMMIT)

## Mesures dans des processus séparés
Chaque processus initialise un seul framework ; il est terminé avant le suivant pour libérer CUDA ou le TPU. Les sorties sont vérifiées avant de rendre les mesures classables.

In [ ]:
helper = REPO / "scripts/benchmarks/colab_micro_suite.py"
helper.parent.mkdir(parents=True, exist_ok=True)
helper.write_text(
    """
GPU_RUNTIMES = ("cuda", "vulkan", "opencl")

def contenders(frameworks, backend, runtimes):
    if backend not in ("cpu", "cuda", "tpu"):
        raise ValueError("Invalid benchmark backend")
    if len(set(frameworks)) != len(frameworks):
        raise ValueError("Duplicate framework")
    if len(set(runtimes)) != len(runtimes) or any(
        r not in GPU_RUNTIMES for r in runtimes
    ):
        raise ValueError("Invalid or duplicate GPU runtime")
    if backend == "cuda" and "mimir" in frameworks and not runtimes:
        raise ValueError("No validated native Mimir GPU runtime")
    result = []
    for framework in frameworks:
        if framework not in ("mimir", "pytorch", "tensorflow", "jax"):
            raise ValueError("Invalid framework")
        if framework == "mimir" and backend == "cuda":
            result.extend(
                dict(label="mimir-" + r, framework="mimir", runtime=r)
                for r in runtimes
            )
        else:
            result.append(dict(
                label=framework, framework=framework, runtime=backend
            ))
    return result

def validate_identity(record, contender, backend, case):
    if (record.get("framework"), record.get("backend"), record.get("case")) != (
        contender["framework"], backend, case
    ):
        raise RuntimeError("Incorrect benchmark result identity")
    if contender["framework"] == "mimir":
        if record.get("runtime") != contender["runtime"].upper():
            raise RuntimeError(
                "Wrong native Mimir runtime: " + str(record.get("runtime"))
            )
""",
    encoding="utf-8",
)
print("Helper installé :", helper)

In [ ]:
if not VALIDATED:raise RuntimeError('Valider le backend Mímir avant le benchmark')
import random
import importlib.util
micro_spec=importlib.util.spec_from_file_location('mimir_colab_micro_suite',REPO/'scripts/benchmarks/colab_micro_suite.py')
MICRO_SUITE=importlib.util.module_from_spec(micro_spec);micro_spec.loader.exec_module(MICRO_SUITE)
MICRO_ENTRIES=MICRO_SUITE.contenders(FRAMEWORKS,BACKEND,MIMIR_GPU_RUNTIMES)
MICRO_BY_LABEL={entry['label']:entry for entry in MICRO_ENTRIES}
MICRO_FRAMEWORKS=list(MICRO_BY_LABEL)
print('Comparaisons natives :',MICRO_FRAMEWORKS)
source_hashes={str(p.relative_to(REPO)):hashlib.sha256(p.read_bytes()).hexdigest()
    for p in sorted((REPO/'src').rglob('*')) if p.is_file() and p.suffix in {'.cpp','.hpp','.cuh','.cu','.h'}}
manifest=dict(protocol='mimir-colab-micro-v2',launcher_sha256=hashlib.sha256((REPO/'run_mimir.sh').read_bytes()).hexdigest(),environment=ENVIRONMENT,run_id=RUN_ID,created_utc=datetime.now(timezone.utc).isoformat(),backend=BACKEND,
    tpu_target=TPU_TARGET,profile=PROFILE,threads=THREADS,seed=SEED,commit=COMMIT,git_status=DIRTY,
    mimir_version=(REPO/'VERSION').read_text().strip(),cmake_args=list(map(str,CMAKE_ARGS)),cpu_affinity=CPU_AFFINITY,
    host_memory_metric='GNU time process maximum RSS; framework imports included; excludes GPU memory',
    contract='FP32; host inputs/weights to host outputs; synchronization and transfers included; process startup excluded',
    warmup=WARMUP,iterations=ITERATIONS,trials=TRIALS,atol=ATOL,rtol=RTOL,
    platform=platform.platform(),cpu=platform.processor(),frameworks=MICRO_FRAMEWORKS,micro_contenders=MICRO_ENTRIES,
    micro_suite_sha256=hashlib.sha256((REPO/'scripts/benchmarks/colab_micro_suite.py').read_bytes()).hexdigest(),
    device_visibility={k:RUNTIME_ENV.get(k) for k in ('CUDA_VISIBLE_DEVICES','NVIDIA_VISIBLE_DEVICES','PJRT_DEVICE','TPU_VISIBLE_CHIPS')},
    source_sha256=source_hashes,driver_sha256=hashlib.sha256(DRIVER.read_bytes()).hexdigest(),
    worker_sha256=hashlib.sha256(WORKER_PATH.read_bytes()).hexdigest(),
    fixture_sha256={case:hashlib.sha256(p.read_bytes()).hexdigest() for case,p in FIXTURE_PATHS.items()},
    packages={},hardware={})
for framework,python in PYTHONS.items():
    try:manifest['packages'][framework]=run([python,'-m','pip','freeze'],timeout=120).splitlines()
    except Exception as exc:manifest['packages'][framework]=str(exc)
if shutil.which('lscpu'):manifest['hardware']['cpu']=run(['lscpu'],timeout=30)
if BACKEND=='cuda':manifest['hardware']['gpu']=run(['nvidia-smi','--query-gpu=name,uuid,driver_version,memory.total',
                                                        '--format=csv,noheader'],timeout=30)
(RUN_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False))
for filename in ('CMakeCache.txt','compile_commands.json'):
    if (BUILD/filename).is_file():shutil.copy2(BUILD/filename,RUN_DIR/filename)
jobs=[(framework,f,trial) for trial in range(TRIALS) for f in CASES for framework in MICRO_FRAMEWORKS]
random.Random(SEED).shuffle(jobs)
RECORDS=[]
for number,(framework,f,trial) in enumerate(jobs,1):
    entry=MICRO_BY_LABEL[framework];implementation=entry['framework']
    case=f['case'];result_file=RUN_DIR/f'{case}.{framework}.{trial}.json'
    print(f'[{number}/{len(jobs)}] {framework} | {case} | essai {trial+1}',flush=True)
    record=dict(framework=framework,case=case,trial=trial,backend=BACKEND,status='error')
    start=time.perf_counter()
    result_file.unlink(missing_ok=True)
    rss_file=result_file.with_suffix('.rss-kib.txt');rss_file.unlink(missing_ok=True)
    try:
        if implementation in INSTALL_ERRORS:raise RuntimeError(INSTALL_ERRORS[implementation])
        command=(['bash',REPO/'run_mimir.sh',BACKEND,FIXTURE_PATHS[case],result_file] if implementation=='mimir' else
                 [PYTHONS[implementation],WORKER_PATH,implementation,BACKEND,FIXTURE_PATHS[case],result_file])
        command=['/usr/bin/time','-f','%M','-o',str(rss_file),
                 'taskset','-c',','.join(map(str,CPU_AFFINITY)),*command]
        job_environment=dict(RUNTIME_ENV)
        if implementation=='mimir':
            job_environment['MIMIR_BENCH_RUNTIME']=entry['runtime']
            job_environment['MIMIR_EXECUTABLE']=str(BENCH_BINARY.resolve())
            job_environment['GOMP_CPU_AFFINITY']=' '.join(map(str,CPU_AFFINITY))
        run(command,cwd=REPO,env=job_environment,timeout=CASE_TIMEOUT)
        record.update(json.loads(result_file.read_text()))
        record['reported_peak_host_rss_mib']=record.get('peak_host_rss_mib')
        record['peak_host_rss_mib']=float(rss_file.read_text().strip())/1024
        record['host_memory_metric']='gnu-time-process-maxrss'
        if not np.isfinite(record['peak_host_rss_mib']) or record['peak_host_rss_mib']<=0:
            raise RuntimeError('Mesure RSS externe invalide')
        if implementation=='mimir':record['version']=manifest['mimir_version']
        MICRO_SUITE.validate_identity(record,entry,BACKEND,case)
        if record['status']!='measured':raise RuntimeError(record.get('error','Mesure absente')+'\n'+record.get('traceback',''))
        outputs=record.pop('outputs');gold=GOLD[case]
        if len(outputs)!=len(gold):raise RuntimeError('Nombre de sorties différent de la référence')
        errors=[]
        for output_index,(actual,expected) in enumerate(zip(outputs,gold)):
            a=np.asarray(actual,np.float64).reshape(-1);b=np.asarray(expected,np.float64).reshape(-1)
            if a.shape!=b.shape or not np.all(np.isfinite(a)):raise RuntimeError('Shape ou sortie non finie')
            errors.append(float(np.max(np.abs(a-b),initial=0)))
            if not np.allclose(a,b,rtol=RTOL,atol=ATOL):raise RuntimeError(f'Échec numérique ; sortie {output_index} ; erreur max {errors[-1]} ; index {int(np.argmax(np.abs(a-b)))} ; obtenu {a[int(np.argmax(np.abs(a-b)))]} ; attendu {b[int(np.argmax(np.abs(a-b)))]}')
        samples=np.asarray(record['samples_ms'],float)
        if len(samples)!=ITERATIONS or not np.all(np.isfinite(samples)) or np.any(samples<=0):
            raise RuntimeError('Échantillons de chronométrage invalides')
        record.update(status='ok',max_abs_error=max(errors),precision='FP32',contract='host_roundtrip')
    except Exception as exc:record.update(status='error',error=str(exc))
    record['process_wall_ms']=(time.perf_counter()-start)*1000
    record.update(framework=framework,implementation=implementation,requested_runtime=entry['runtime'])
    record['trial']=trial;record.pop('outputs',None)
    result_file.write_text(json.dumps(record,indent=2,allow_nan=False,ensure_ascii=False))
    RECORDS.append(record)
    (RUN_DIR/'records.json').write_text(json.dumps(RECORDS,indent=2,allow_nan=False,ensure_ascii=False))
    print('  ->',record['status'],record.get('error',''),flush=True)
assert manifest['source_sha256']=={str(p.relative_to(REPO)):hashlib.sha256(p.read_bytes()).hexdigest()
    for p in sorted((REPO/'src').rglob('*')) if p.is_file() and p.suffix in {'.cpp','.hpp','.cuh','.cu','.h'}}, 'Sources changées pendant le benchmark'

## Tableau, classement et note Mímir

In [ ]:
import pandas as pd
from IPython.display import display
# Each cell is valid only if every requested independent trial passed.
summary=[]
for framework in MICRO_FRAMEWORKS:
    for f in CASES:
        rows=[r for r in RECORDS if r['framework']==framework and r['case']==f['case']]
        valid=len(rows)==TRIALS and all(r['status']=='ok' for r in rows)
        row=dict(framework=framework,case=f['case'],backend=BACKEND,status='ok' if valid else 'error',
                 trials_ok=sum(r['status']=='ok' for r in rows),trials_requested=TRIALS,
                 median_ms=None,p95_ms=None,iqr_ms=None,throughput_per_s=None,unit=f['unit'],
                 cold_ms=None,setup_ms=None,peak_host_rss_mib=None,max_abs_error=None)
        row.update(version=next((r.get('version','unknown') for r in rows if r['status']=='ok'),'unavailable'),
                   mode=next((r.get('mode','unknown') for r in rows if r['status']=='ok'),'unavailable'),
                   device=next((r.get('device_name',r.get('device',r.get('runtime','unknown'))) for r in rows if r['status']=='ok'),'unavailable'))
        if valid:
            samples=np.concatenate([r['samples_ms'] for r in rows])
            row.update(median_ms=float(np.median(samples)),p95_ms=float(np.percentile(samples,95)),
                iqr_ms=float(np.percentile(samples,75)-np.percentile(samples,25)),
                throughput_per_s=float(f['units']*1000/np.median(samples)),
                cold_ms=float(np.median([r['cold_ms'] for r in rows])),
                setup_ms=float(np.median([r['setup_ms'] for r in rows])),
                peak_host_rss_mib=max(r['peak_host_rss_mib'] for r in rows),
                max_abs_error=max(r['max_abs_error'] for r in rows))
        else:row['error']=' | '.join(sorted(set(r.get('error','missing result') for r in rows if r['status']!='ok'))) or 'missing trial'
        summary.append(row)
detail=pd.DataFrame(summary)
eligible=[name for name in MICRO_FRAMEWORKS if all(r['status']=='ok' for r in summary if r['framework']==name)
          and sum(r['framework']==name for r in summary)==len(CASES)]
# Fastest valid latency among COMPLETE contenders is 100 points per workload.
# Geometric mean gives equal weight to each declared workload. No missing case
# is imputed, and failures cannot improve a score by shrinking the denominator.
leaderboard=[]
for framework in MICRO_FRAMEWORKS:
    rows=[r for r in summary if r['framework']==framework]
    complete=framework in eligible
    score=None
    if complete and len(eligible)>=2:
        ratios=[]
        for row in rows:
            best=min(s['median_ms'] for s in summary if s['case']==row['case'] and s['framework'] in eligible)
            ratios.append(best/row['median_ms'])
        score=float(100*np.exp(np.mean(np.log(ratios))))
    leaderboard.append(dict(framework=framework,backend=BACKEND,coverage=f"{sum(r['status']=='ok' for r in rows)}/{len(CASES)}",
        score_100=score,rank=None,status='classé' if score is not None else 'incomplet' if not complete else 'pas assez de concurrents',
        latency_metric='geometric mean of per-case relative median latency'))
board=pd.DataFrame(leaderboard).sort_values('score_100',ascending=False,na_position='last').reset_index(drop=True)
scored=board['score_100'].notna();board.loc[scored,'rank']=board.loc[scored,'score_100'].rank(method='min',ascending=False).astype(int)
display(board[['rank','framework','backend','coverage','score_100','status']])
with pd.option_context('display.max_colwidth', None):
    display(detail)
# Plain text preserves tracebacks when Colab truncates the interactive table.
failed_records=[r for r in RECORDS if r['status']!='ok']
errors_text='\n\n'.join(
    f"{r['framework']} | {r['case']} | essai {r.get('trial',0)+1}\n{r.get('error','missing result')}"
    for r in failed_records)
(RUN_DIR/'bench-errors.txt').write_text(errors_text)
if errors_text:
    print('Erreurs complètes (également dans bench-errors.txt) :\n'+errors_text)
for _,mimir_row in board[board.framework.str.startswith('mimir')].iterrows():
    if pd.notna(mimir_row.score_100):print(f"{mimir_row.framework} : rang {int(mimir_row['rank'])}/{len(eligible)}, note {mimir_row.score_100:.2f}/100 sur ce profil.")
    else:print(mimir_row.framework,': aucune note fiable sur ce run ; consulter les erreurs et la couverture.')
print('La note mesure ce profil, ce matériel et cette frontière hôte ; elle ne note pas la qualité générale des frameworks.')
detail.to_csv(RUN_DIR/'bench-detail.csv',index=False);board.to_csv(RUN_DIR/'bench-leaderboard.csv',index=False)
html='<html><head><meta charset="utf-8"><title>Mímir benchmark</title></head><body>'
html+=f'<h1>Benchmark {RUN_ID}</h1><p>FP32 / {BACKEND} / {PROFILE} / host roundtrip / seed {SEED}</p>'
html+='<p>Score = 100 × moyenne géométrique(best median / framework median), uniquement pour les concurrents complets. '
html+='Au moins deux concurrents complets requis ; les erreurs restent non classées.</p>'
html+=board.to_html(index=False,escape=True)+detail.to_html(index=False,escape=True)+'</body></html>'
(RUN_DIR/'report.html').write_text(html)

## Diagnostic natif : modèles complets Linear/SiLU et VGG16

Cette section mesure les nouvelles optimisations **Mímir seul**, sans téléchargement de données : forward, loss, backward et mise à jour SGD/AdamW/Lion. Elle ne participe ni au classement entre frameworks, ni à la note MNIST.

- `quick` : petit Linear/SiLU et VGG 64×64 (base 4).
- `standard` : deux tailles Linear/SiLU et VGG 128×128 / 512×512 (base 16, GroupNorm, Lion).
- Latence froide (calibration comprise), médiane/p95 chaudes et échantillons bruts séparés ; mémoire RSS+swap et temps du processus limités.
- Le lanceur de production et le planner `cost` restent actifs : un GPU disponible ne garantit pas que chaque couche l'utilise. Les journaux conservent les décisions mesurées.
- Les arrondis float32 peuvent modifier la trajectoire de Lion ; aucune identité bit à bit des poids n'est promise. Ce diagnostic n'est pas exécuté sur TPU.


In [ ]:
RUN_NATIVE_BENCH=True
NATIVE_PROFILE=PROFILE
NATIVE_TRIALS=1 if NATIVE_PROFILE=='quick' else 2
NATIVE_TIMEOUT_S=300
NATIVE_HOST_MEMORY_MIB=4096
NATIVE_RECORDS=[]
if RUN_NATIVE_BENCH and BACKEND=='tpu':
    print('Diagnostic natif Linear/VGG non exécuté sur TPU ; les benchmarks TPU existants restent disponibles.')
elif RUN_NATIVE_BENCH:
    if not VALIDATED:raise RuntimeError('Valider le runtime Mímir avant le diagnostic natif')
    import importlib.util
    native_source=REPO/'scripts/benchmarks/colab_native_suite.py'
    if not native_source.is_file():raise RuntimeError('Instantané trop ancien : recharger ce notebook et extraire ses sources actuelles.')
    native_spec=importlib.util.spec_from_file_location('mimir_colab_native',native_source)
    NATIVE_SUITE=importlib.util.module_from_spec(native_spec);native_spec.loader.exec_module(NATIVE_SUITE)
    NATIVE_CASES=NATIVE_SUITE.cases(NATIVE_PROFILE)
    NATIVE_DIR=RUN_DIR/('native-whole-model-'+os.urandom(3).hex());NATIVE_DIR.mkdir()
    native_targets=sorted({case['target'] for case in NATIVE_CASES})
    run(['cmake','--build',BUILD,'-j',JOBS,'--target',*native_targets],timeout=2400)
    native_paths=['run_mimir.sh','src/Model.cpp','src/Planning/ExecutionCalibration.cpp',
        'src/runtimes/cpu/ConvolutionKernels.hpp','src/runtimes/cpu/HardwareOpt.hpp',
        'src/runtimes/cpu/LayerOps.hpp','src/runtimes/cpu/RuntimeLayerDispatch.hpp',
        'src/runtimes/portable/PortableDevice.cpp','src/runtimes/portable/PortableStaging.hpp',
        'scripts/benchmarks/colab_native_suite.py','scripts/benchmarks/colab_real_suite.py',
        'scripts/benchmarks/benchmark_whole_model.cpp','scripts/benchmarks/benchmark_vgg16_whole_model.cpp']
    native_hashes={name:hashlib.sha256((REPO/name).read_bytes()).hexdigest() for name in native_paths}
    native_protocol=dict(kind='mimir_native_diagnostic',participates_in_comparative_score=False,
        profile=NATIVE_PROFILE,cases=NATIVE_CASES,trials=NATIVE_TRIALS,backend_requested=BACKEND,
        gpu_runtimes_validated=list(MIMIR_GPU_RUNTIMES),planner_mode='cost',allow_cpu_fallback=True,
        threads=THREADS,cpu_affinity=CPU_AFFINITY,process_timeout_s=NATIVE_TIMEOUT_S,
        host_rss_plus_swap_limit_mib=NATIVE_HOST_MEMORY_MIB,warmup_steps=dict(linear=10,vgg16=2),
        source_snapshot_sha256=globals().get('SOURCE_SHA256'),sources=native_hashes,
        contract='native public whole-model APIs; synthetic input; no GUI or dataset decoding; production launcher; cold and warm timings separated')
    (NATIVE_DIR/'protocol.json').write_text(json.dumps(native_protocol,indent=2))
    NATIVE_RECORDS=NATIVE_SUITE.run_suite(REPO,BUILD,NATIVE_DIR,NATIVE_CASES,RUNTIME_ENV,CPU_AFFINITY,
        trials=NATIVE_TRIALS,timeout_s=NATIVE_TIMEOUT_S,memory_mib=NATIVE_HOST_MEMORY_MIB)
    assert native_hashes=={name:hashlib.sha256((REPO/name).read_bytes()).hexdigest() for name in native_paths}, 'Sources modifiées pendant le diagnostic'
    native_rows=[]
    for row in NATIVE_RECORDS:
        result=row.get('measurement',{});training=result.get('training',result.get('total',{}))
        native_rows.append(dict(case=row['case'],trial=row['trial'],status=row['status'],
            cold_ms=result.get('cold_trainstep_ms'),train_median_ms=training.get('median_ms'),train_p95_ms=training.get('p95_ms'),
            forward_ms=result.get('forward',{}).get('median_ms'),backward_loss_ms=result.get('backward_and_loss',{}).get('median_ms'),
            optimizer_ms=result.get('optimizer_step',{}).get('median_ms'),inference_ms=result.get('inference',{}).get('median_ms'),
            final_loss=result.get('final_loss'),peak_rss_swap_mib=row.get('peak_tree_rss_swap_mib'),log=row['log']))
    NATIVE_TABLE=pd.DataFrame(native_rows);NATIVE_TABLE.to_csv(NATIVE_DIR/'summary.csv',index=False)
    display(NATIVE_TABLE)
    print('Diagnostic natif (hors classement comparatif) :',NATIVE_DIR)


## Usage réel : entraînement et inférence MNIST sous contraintes

Ce second tableau mesure une utilisation complète des API : **Mímir `Model.trainStep`/`forwardPass`**, PyTorch `nn.Sequential`/autograd/SGD/Adam/AdamW, Keras `Sequential`/`GradientTape`/SGD/Adam/AdamW et JAX `jit(value_and_grad)`. Les poids persistent entre les mises à jour. Il ne réinitialise pas le modèle à chaque pas.

- MLP `784 → 128 → ReLU → 10`, FP32, MSE sur labels one-hot, SGD sans momentum, Adam, AdamW avec décroissance découplée .01. Même initialisation, mêmes images, mêmes batches cycliques après un mélange initial commun. Batchs comparés séparément ; aucun score ne mélange des charges différentes au sein d'un cas.
- MNIST est téléchargé et contrôlé par le [SHA256 publié par Keras](https://github.com/keras-team/keras/blob/master/keras/src/datasets/mnist.py). Le train officiel est conservé ; les 2 000 premières images du test officiel servent à la validation périodique, les 8 000 autres à l'évaluation finale. Le profil `quick` utilise des sous-ensembles et sert au contrôle fonctionnel.
- **Contraintes appliquées** : même affinité et nombre de threads CPU, plafond mémoire hôte RSS+swap cumulé du processus et de ses workers, délai total par essai (imports, compilation et données compris). Un watchdog externe termine tout le groupe de processus en cas de dépassement. Échantillonnage à 50 ms : une pointe entre deux mesures peut échapper au moniteur ; les pages partagées peuvent être comptées plusieurs fois. Pas de quota VRAM/HBM artificiel ; les limites physiques et OOM restent visibles.
- Les étapes chaudes incluent préparation du batch, transferts, forward, loss, backward, SGD/Adam/AdamW et synchronisation. Les frameworks conservent leurs paramètres selon leur API native. Chez Mímir, la loss reste côté hôte et les optimiseurs passent par le routeur de runtimes et les transferts du runtime TPU sont inclus ; le planner `cost` peut choisir des routes CPU et accélérateur, comme en utilisation réelle. Fusions, réutilisation des buffers et résidence sur le device sont autorisées ; la compilation native active SIMD et OpenMP. Les plans et la trace de diagnostic indiquent ce qui est applicable : une option activée ne garantit pas une fusion de kernel ou une résidence effective sur chaque runtime. Les fusions d'entraînement restent limitées à celles prises en charge par le framework. Ce scénario mesure ces chemins publics et ce MLP ; il ne résume pas tous les usages ou optimisations des frameworks.
- Première étape vérifiée contre une référence NumPy float64 (prédictions, loss et sondes de paramètres après update, biais inclus). Une précision minimale sur le test tenu à part est obligatoire pour classer un essai. L'entraînement continue jusqu'au travail fixe demandé, même si le seuil de validation est atteint.
- Mesures : débit d'entraînement **validation périodique incluse**, débit d'inférence, p50/p95/IQR des étapes, latences à froid, temps total, pic mémoire du groupe, précision/MSE et temps observé pour atteindre la précision cible. Les deux modes s'évaluent sur le même unique accélérateur, dans des processus isolés et un ordre aléatoire.
- La note d'usage réel est indépendante de celle des opérations : moyenne géométrique des débits relatifs au meilleur, poids égaux entre entraînement, inférence et batchs. Tous les cas et essais doivent passer les budgets, la vérification numérique et le seuil de qualité ; il faut deux frameworks complets. Dépassements, OOM, erreurs et qualité insuffisante restent non classés. Les médianes et p95 ne constituent pas à eux seuls une preuve de significativité statistique.

Les essais croisent chaque batch avec SGD, Adam et AdamW. Les états des optimiseurs sont remis à zéro après les passes froides. Les taux sont communs aux frameworks pour chaque optimiseur (.2 / .001 / .001), AdamW utilise une décroissance .01. Initialisation (import/runtime/modèle), allocation dédiée, premier pas, durée totale du processus et RSS sont exportés séparément. La note existante mesure le débit ; les coûts froids et mémoire restent des axes de comparaison distincts.

La sonde d’allocation remplit 16 Mio et attend la fin du travail. Elle mesure le chemin public (cache éventuel compris), sans prétendre isoler un appel système. Mímir alloue les tenseurs publics en RAM ; ses allocations accélérateur différées sont incluses dans le premier pas. JAX inclut le staging hôte. Le delta RSS de la sonde peut être nul ou négatif ; le pic RSS surveillé couvre aussi les processus enfants. RSS ne mesure pas la VRAM/HBM. Les budgets RAM et délai sont ajustés automatiquement à la machine et au backend.

Un second score `lifecycle_score_100` compare à parts égales le temps total du processus et son pic RSS (moyenne géométrique des ratios au meilleur, par cas). Il intègre les coûts froids dans le temps total, conserve les contrôles numériques et de qualité, et exclut les frameworks incomplets. Le score de débit existant reste affiché séparément. La sonde d’allocation hôte/appareil est descriptive et ne participe pas directement au score.

Le profil MNIST suit automatiquement `PROFILE`. Le temps processus inclut aussi les vérifications propres aux adaptateurs et le diagnostic de routes Mímir ; le détail brut permet de distinguer ces coûts. L’installation des dépendances et le build sont hors des essais. La détection `auto` choisit le GPU NVIDIA disponible, sinon le TPU, sinon le CPU : une session Colab ne fournit pas nécessairement les trois matériels.


In [ ]:
# Real workload: independent from the microbenchmark score.
RUN_REAL_BENCH=True
REAL_PROFILE=PROFILE  # quick = smoke on real data; standard = full MNIST split
REAL_TRIALS=2 if REAL_PROFILE=='standard' else 1
REAL_BATCHES=[32,128,512] if REAL_PROFILE=='standard' else [64]
REAL_HIDDEN=128
REAL_STEPS=400 if REAL_PROFILE=='standard' else 80
REAL_EVAL_EVERY=50 if REAL_PROFILE=='standard' else 40
REAL_OPTIMIZERS={'sgd':dict(lr=.2,weight_decay=0.),'adam':dict(lr=.001,weight_decay=0.),'adamw':dict(lr=.001,weight_decay=.01)}
REAL_TARGET_ACCURACY=.85
REAL_MIN_TEST_ACCURACY=.75 if REAL_PROFILE=='standard' else .4
REAL_PROCESS_TIMEOUT_S=600 if BACKEND=='tpu' else 300
# Leave RAM for the notebook/build; apply the same limit to all frameworks.
_host_meminfo=dict(line.split(':',1) for line in Path('/proc/meminfo').read_text().splitlines())
_available_mib=int(_host_meminfo['MemAvailable'].split()[0])/1024
REAL_HOST_MEMORY_MIB=max(512,min(12288,int(_available_mib*.65)))  # monitored sum RSS+swap across process and its workers
REAL_MIMIR_OPTIONS=dict(planner_mode='cost',fusion=True,fusion_train=True,
    buffer_reuse=True,device_residency=True,allow_cpu_fallback=True)
assert REAL_PROFILE in {'quick','standard'} and REAL_TRIALS>=1
assert all(isinstance(b,int) and b>0 for b in REAL_BATCHES) and len(set(REAL_BATCHES))==len(REAL_BATCHES)
assert REAL_STEPS>0 and REAL_EVAL_EVERY>0 and REAL_HIDDEN>0
assert 0<=REAL_MIN_TEST_ACCURACY<=1 and 0<=REAL_TARGET_ACCURACY<=1
assert REAL_PROCESS_TIMEOUT_S>0 and REAL_HOST_MEMORY_MIB>0
assert REAL_OPTIMIZERS and set(REAL_OPTIMIZERS)<=set(['sgd','adam','adamw'])
assert all(o['lr']>0 and o['weight_decay']>=0 for o in REAL_OPTIMIZERS.values())


### Données communes, modèle natif et provenance


In [ ]:
if RUN_REAL_BENCH:
    import importlib.util
    real_source=REPO/'scripts/benchmarks'
    required_real=['colab_real_suite.py','colab_real_worker.py','colab_real_model.cpp']
    missing_real=[name for name in required_real if not (real_source/name).is_file()]
    if missing_real:
        raise RuntimeError('Ancien instantane Mimir : '+str(missing_real)+'. Recharger la version actuelle du notebook puis relancer depuis la premiere cellule (configuration et extraction des sources comprises).')
    spec=importlib.util.spec_from_file_location('mimir_colab_real_suite',real_source/'colab_real_suite.py')
    REAL_SUITE=importlib.util.module_from_spec(spec);spec.loader.exec_module(REAL_SUITE)
    REAL_DIR=RUN_DIR/('real-workload-'+datetime.now(timezone.utc).strftime('%H%M%S')+'-'+os.urandom(3).hex());REAL_DIR.mkdir(exist_ok=False)
    REAL_DATA=REAL_DIR/'data'
    REAL_DATA_MANIFEST=REAL_SUITE.prepare_data(REAL_DATA,[REAL_HIDDEN],SEED,
        quick=REAL_PROFILE=='quick',archive=Path('/content/mimir-bench-cache/mnist.npz'))
    sizes=[REAL_HIDDEN*784,REAL_HIDDEN,10*REAL_HIDDEN,10]
    probe_indices=[];offset=0
    for size in sizes:
        probe_indices.extend((offset+np.unique(np.linspace(0,size-1,min(32,size),dtype=int))).tolist());offset+=size
    REAL_CASES=[dict(case=f'mnist-mlp-{REAL_HIDDEN}-batch-{batch}-{optimizer}',optimizer=optimizer,weight_decay=options['weight_decay'],backend=BACKEND,
        data_dir=str(REAL_DATA),mimir_version=(REPO/'VERSION').read_text().strip(),hidden=REAL_HIDDEN,batch=batch,steps=REAL_STEPS,lr=options['lr'],
        eval_every=REAL_EVAL_EVERY,target_accuracy=REAL_TARGET_ACCURACY,probe_indices=probe_indices,mimir=REAL_MIMIR_OPTIONS,
        train_examples=(REAL_DATA/'train.labels.u8').stat().st_size,
        test_examples=(REAL_DATA/'test.labels.u8').stat().st_size) for batch in REAL_BATCHES for optimizer,options in REAL_OPTIMIZERS.items()]
    # Add a native public-Model driver to the existing deferred CMake hook.
    real_hook='\nfunction(mimir_colab_add_real_benchmark)\n  if(TARGET mimir_core AND NOT TARGET mimir_colab_real)\n    add_executable(mimir_colab_real "${CMAKE_SOURCE_DIR}/scripts/benchmarks/colab_real_model.cpp")\n    target_compile_features(mimir_colab_real PRIVATE cxx_std_17)\n    target_link_libraries(mimir_colab_real PRIVATE mimir_core)\n    set_target_properties(mimir_colab_real PROPERTIES RUNTIME_OUTPUT_DIRECTORY "${CMAKE_BINARY_DIR}/bin")\n  endif()\nendfunction()\ncmake_language(DEFER CALL mimir_colab_add_real_benchmark)\n'
    original_hook=INJECT.read_text()
    if 'function(mimir_colab_add_real_benchmark)' not in original_hook:INJECT.write_text(original_hook+real_hook)
    run(CMAKE_ARGS+['-DCMAKE_PROJECT_INCLUDE='+str(INJECT)])
    run(['cmake','--build',BUILD,'-j',JOBS,'--target','mimir_colab_real'],timeout=2400)
    REAL_BINARY=BUILD/'bin/mimir_colab_real'
    REAL_WORKER=REAL_DIR/'sources/colab_real_worker.py'
    provenance=dict(launcher_sha256=REAL_SUITE.sha256(REPO/'run_mimir.sh'),profile=REAL_PROFILE,trials=REAL_TRIALS,cases=REAL_CASES,
        process_timeout_s=REAL_PROCESS_TIMEOUT_S,host_rss_plus_swap_limit_mib=REAL_HOST_MEMORY_MIB,
        min_test_accuracy=REAL_MIN_TEST_ACCURACY,cpu_affinity=CPU_AFFINITY,threads=THREADS,
        seed=SEED,dataset=REAL_DATA_MANIFEST,source_snapshot_sha256=globals().get('SOURCE_SHA256'),
        sources={p.name:REAL_SUITE.sha256(p) for p in real_source.glob('colab_real_*') if p.is_file()},
        contract='persistent weights; public whole-model APIs; host batch preprocessing and device transfer; synchronized SGD/Adam/AdamW; fixed validation schedule; final held-out inference',
        fp32=True,tf32=False,mimir_options=REAL_MIMIR_OPTIONS,cmake_args=list(map(str,CMAKE_ARGS)),
        mimir_layers='production planner; CPU/native accelerator placement allowed; native host loss; optimizer dispatched by runtime router; see plans and runtime audit',
        gpu_memory_limit='physical capacity only; no artificial VRAM/HBM cap',memory_poll_s=.05)
    (REAL_DIR/'protocol.json').write_text(json.dumps(provenance,indent=2))
    (REAL_DIR/'sources').mkdir(exist_ok=True)
    for p in real_source.glob('colab_real_*'):
        if p.is_file():shutil.copy2(p,REAL_DIR/'sources'/p.name)
    print('Usage reel :',len(REAL_CASES),'cas,',REAL_TRIALS,'essais,',REAL_STEPS,'updates/cas ;',REAL_DIR)


### Essais isolés sous budgets effectifs


In [ ]:
if RUN_REAL_BENCH:
    if not VALIDATED:raise RuntimeError('Validation Mimir requise avant les essais reels')
    REAL_RECORDS=REAL_SUITE.run_suite(REAL_DIR,REAL_CASES,FRAMEWORKS,PYTHONS,REAL_BINARY,REAL_WORKER,
        RUNTIME_ENV,CPU_AFFINITY,REAL_TRIALS,REAL_PROCESS_TIMEOUT_S,REAL_HOST_MEMORY_MIB,
        REAL_MIN_TEST_ACCURACY,SEED,INSTALL_ERRORS)
    assert provenance['sources']=={p.name:REAL_SUITE.sha256(p) for p in real_source.glob('colab_real_*') if p.is_file()}, 'Sources du benchmark modifiees pendant les essais'
    assert REAL_DATA_MANIFEST['files']=={p.name:REAL_SUITE.sha256(p) for p in sorted(REAL_DATA.iterdir()) if p.suffix in {'.u8','.f32'}}, 'Donnees modifiees pendant les essais'


### Comparaison d’usage réel et écarts avec Mímir


In [ ]:
if RUN_REAL_BENCH:
    REAL_DETAIL,REAL_BOARD=REAL_SUITE.summarize(REAL_RECORDS,REAL_CASES,FRAMEWORKS,REAL_TRIALS)
    display(REAL_BOARD[['rank','framework','coverage','score_100','lifecycle_rank','lifecycle_score_100','status']])
    with pd.option_context('display.max_colwidth',None):display(REAL_DETAIL)
    REAL_DETAIL.to_csv(REAL_DIR/'real-detail.csv',index=False)
    REAL_BOARD.to_csv(REAL_DIR/'real-leaderboard.csv',index=False)
    errors='\n\n'.join(f"{r['framework']} | {r['case']} | essai {r['trial']+1}\n{r.get('error','')}\n{r.get('log_tail','')}"
        for r in REAL_RECORDS if r['status']!='ok')
    (REAL_DIR/'errors.txt').write_text(errors)
    if errors:print(errors)
    report='<html><head><meta charset="utf-8"></head><body><h1>Usage reel sous contraintes</h1>'
    report+='<p>Score = moyenne geometrique des debits relatifs au meilleur, entrainement et inference, poids egaux par batch. Qualite minimale et budgets obligatoires sur tous les essais. Ce score est distinct des microbenchmarks.</p>'
    report+=REAL_BOARD.to_html(index=False,escape=True)+REAL_DETAIL.to_html(index=False,escape=True)+'</body></html>'
    (REAL_DIR/'report.html').write_text(report)
    print('Rapport usage reel :',REAL_DIR/'report.html')
    print('Speedup_vs_mimir > 1 : le concurrent est plus rapide ; < 1 : Mimir est plus rapide.')
    print('time_to_goal absent : seuil non atteint aux evaluations observees ; jamais impute a zero.')
    valid=REAL_DETAIL[REAL_DETAIL.status=='ok']
    if not valid.empty:
        import matplotlib.pyplot as plt
        fig,axes=plt.subplots(1,2,figsize=(13,5))
        for ax,metric,title in zip(axes,['train_examples_per_s','inference_examples_per_s'],['Entrainement + validation','Inference tenue a part']):
            valid.pivot(index='case',columns='framework',values=metric).plot.bar(ax=ax,logy=True)
            ax.set_title(title);ax.set_ylabel('Images/s (log)')
        plt.tight_layout();plt.savefig(REAL_DIR/'throughput.png',dpi=160);plt.show()
    # The existing final archive includes this entire subdirectory and its logs.


## Visualisation et export
Les résultats bruts et leur provenance sont conservés avec le rapport ; ils permettent de vérifier ou recalculer la note.

In [ ]:
import matplotlib.pyplot as plt
valid=detail[detail.status=='ok']
if not valid.empty:
    ax=valid.pivot(index='case',columns='framework',values='median_ms').plot.bar(figsize=(12,5),logy=True)
    ax.set_ylabel('Latence médiane par opération (ms, échelle log)')
    ax.set_title(f'{BACKEND} — FP32 — transferts hôte et synchronisation inclus')
    plt.tight_layout();plt.savefig(RUN_DIR/'latencies.png',dpi=160);plt.show()
archive=shutil.make_archive(str(RUN_DIR), 'zip',root_dir=RUN_DIR)
print('Rapport HTML :',RUN_DIR/'report.html')
print('Archive (manifest, versions, sources hashées, fixtures, mesures brutes, CSV, graphique) :',archive)
# Optional download:
# from google.colab import files
# files.download(archive)